# ATRA-4B — evaluation of the trained adapter

Run 6 of `atra-4b-qlora` trained successfully on 2026-09-20 (200 steps, final
loss 0.206, manifest `status: completed`). The repository labels the model
**UNTRAINED** and the rule the project set for itself is that the label only
changes once `evaluate.py` has run against the *served* model and the
thresholds in `config/default.yaml` pass.

## Why this kernel is running a second time

The first run of this notebook failed, and the way it failed pointed at the
harness rather than the model: `tool_selection_accuracy` and
`tool_argument_validity` were 0/8 exactly, and `unsupported_chain_rejection`
0/5. `train.py` renders every training prompt with
`apply_chat_template(messages, tools=...)`, so all 800 training examples
carried the four tool schemas in the system block — and `evaluate.py` posted
only `messages`, then scored the reply against `example.tools` anyway. The
model was judged on a prompt shape it had never seen.

`evaluate.py` is fixed: `EndpointModel.generate` now sends the tools with the
prompt in the same OpenAI shape `train.py` builds. Nothing else changed — not
the thresholds, not the sampling, not the test split. The probe cell now
sends the same body the harness sends, and prints the prompt length with and
without the tool block, because that difference is the direct evidence that
the fix took effect.

That check earned its keep immediately: attempt 2 stopped at it, because the
GGUF carries no chat template and llama-server was rendering a default one
with no `tools` block, so the schemas were dropped after `evaluate.py` sent
them. Section 3.1 recovers the training tokenizer's template and serves it.
The weights are untouched.

A failure is a result too, and it gets reported as it comes.

This kernel produces that result, end to end, in one run:

1. pinned dependencies, minus torchao;
2. the trained adapter staged from the dataset `atra12/atra-4b-adapter-v0`
   (nothing is retrained);
3. merged into `Qwen/Qwen3-4B-Instruct-2507` and converted to GGUF q4_k_m;
4. served by `llama-server` on an OpenAI-compatible endpoint;
5. the test split rebuilt with the exact command the thresholds were written
   for, `python -m data.build --seed 42 --per-domain 200 --out data/out`;
6. `evaluate.py` run against the local endpoint, its JSON saved, and every
   metric printed against its threshold with the exit code.

A failing evaluation is a result. Nothing in this notebook is tuned to make
the numbers look better, and the exit code is reported as it comes.

In [ ]:
!nvidia-smi
import platform, subprocess, torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("python", platform.python_version())
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True).stdout)
!ls -la /kaggle/input/*
!df -h

## 1. Dependencies

The training run's pins, minus torchao.

In [ ]:
%%bash
set -e
set -o pipefail
pip install -q "transformers==4.57.6" "peft==0.19.1" "accelerate==1.12.0" \
  "huggingface_hub==0.36.2" "sentencepiece==0.2.1" "pyyaml==6.0.3"
# Kaggle ships torchao 0.10.0. peft's LoRA dispatcher calls
# is_torchao_available() for every module it injects, and that helper raises
# on a version below 0.16.0 rather than returning False. The merge loads the
# base in fp16, so it reaches that dispatcher. Neither path uses torchao.
pip uninstall -q -y torchao || true
python - <<'PY'
import importlib.util
print("torchao present:", importlib.util.find_spec("torchao") is not None)
import accelerate, peft, transformers, yaml
print("deps ok:", transformers.__version__, peft.__version__, accelerate.__version__)
PY

## 2. The repository files

`evaluate.py`, the `data` package, `config/default.yaml` and `export.py`,
carried verbatim rather than cloned. The thresholds this run is judged
against are the ones in the file printed below, and a digest per file makes
that checkable against the working tree.

In [ ]:
import base64, hashlib, pathlib, shutil

# The ATRA repository files this evaluation needs, carried verbatim as base64.
# Carried rather than cloned so the run cannot silently evaluate a different
# revision of evaluate.py, data/build.py or config/default.yaml than the one
# whose thresholds this result will be reported against. Compare each digest
# below with `sha256sum` on the same file in model/atra-4b/.
FILES = {
    "evaluate.py": (
        "..."
    ),
    "export.py": (
        "IiIiTWVyZ2UgYSB0cmFpbmVkIEFUUkEtNEIgTG9SQSBhZGFwdGVyIGludG8gaXRzIGJhc2Ug"
        "bW9kZWwgYW5kIGV4cG9ydCBpdC4NCg0KVHdvIG91dHB1dHMsIGJvdGggb3B0aW9uYWw6DQoN"
        "CjEuIGBgPG91dD4vbWVyZ2VkYGAg4oCUIHRoZSBiYXNlIG1vZGVsIHdpdGggdGhlIGFkYXB0"
        "ZXIgZm9sZGVkIGluLCBzYXZlZCBhcw0KICAgc2FmZXRlbnNvcnMgaW4gZnAxNiAob3IgYmYx"
        "NiBvbiBoYXJkd2FyZSB0aGF0IHN1cHBvcnRzIGl0KS4gVGhpcyBpcyB3aGF0IGENCiAgIEdH"
        "VUYgY29udmVydGVyIGNvbnN1bWVzLg0KMi4gYGA8b3V0Pi9hdHJhLTRiLTx0eXBlPi5nZ3Vm"
        "YGAg4oCUIGEgbGxhbWEuY3BwIEdHVUYgZmlsZSwgcHJvZHVjZWQgYnkgdGhlDQogICBgYGNv"
        "bnZlcnRfaGZfdG9fZ2d1Zi5weWBgIHNjcmlwdCBvZiBhIGxsYW1hLmNwcCBjaGVja291dCB0"
        "aGF0IHRoZSBjYWxsZXINCiAgIHBvaW50cyBhdCB3aXRoIGBgLS1sbGFtYS1jcHBgYC4gYGBm"
        "MTZgYCBhbmQgYGBxOF8wYGAgbmVlZCBvbmx5IHRoYXQgUHl0aG9uDQogICBzY3JpcHQ7IGBg"
        "cTRfa19tYGAgYWRkaXRpb25hbGx5IG5lZWRzIGEgYnVpbHQgYGBsbGFtYS1xdWFudGl6ZWBg"
        "IGJpbmFyeSwNCiAgIHdoaWNoIGlzIGxvb2tlZCBmb3IgbmV4dCB0byB0aGUgY2hlY2tvdXQn"
        "cyBgYGJ1aWxkL2JpbmBgLg0KDQpOb3RoaW5nIGhlcmUgdHJhaW5zLCBldmFsdWF0ZXMgb3Ig"
        "dXBsb2Fkcy4gSXQgd3JpdGVzIGBgZXhwb3J0LW1hbmlmZXN0Lmpzb25gYA0KbmV4dCB0byB0"
        "aGUgb3V0cHV0cyBzbyBhIEdHVUYgY2FuIGFsd2F5cyBiZSB0cmFjZWQgYmFjayB0byB0aGUg"
        "YWRhcHRlciwgdGhlDQpiYXNlIHJldmlzaW9uIGFuZCB0aGUgdHJhaW5pbmcgbWFuaWZlc3Qg"
        "dGhhdCBwcm9kdWNlZCBpdCDigJQgYSBmaWxlIHRoYXQgbGFja3MNCnRoYXQgY2hhaW4gaXMg"
        "bm90IGFuIEFUUkEtNEIgcmVsZWFzZSwgd2hhdGV2ZXIgaXQgaXMgY2FsbGVkLg0KDQpVc2Fn"
        "ZTo6DQoNCiAgICBweXRob24gZXhwb3J0LnB5IC0tYWRhcHRlciBydW5zL2F0cmEtNGIgLS1v"
        "dXQgZXhwb3J0DQogICAgcHl0aG9uIGV4cG9ydC5weSAtLWFkYXB0ZXIgcnVucy9hdHJhLTRi"
        "IC0tb3V0IGV4cG9ydCBcDQogICAgICAgIC0tbGxhbWEtY3BwIC9rYWdnbGUvd29ya2luZy9s"
        "bGFtYS5jcHAgLS1nZ3VmIHE4XzANCg0KVGhlIGFkYXB0ZXIgZGlyZWN0b3J5IG11c3QgY29u"
        "dGFpbiBgYG1hbmlmZXN0Lmpzb25gYCBmcm9tIGBgdHJhaW4ucHlgYCB3aXRoDQpgYHN0YXR1"
        "czogY29tcGxldGVkYGAuIEEgc21va2UgbWFuaWZlc3QgaXMgcmVmdXNlZDogYSAzMC1zdGVw"
        "IGNoZWNrcG9pbnQgaXMgbm90DQphIG1vZGVsLCBhbmQgZXhwb3J0aW5nIGl0IHdvdWxkIG9u"
        "bHkgbWFrZSBpdCBsb29rIGxpa2Ugb25lLg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBv"
        "cnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgaGFzaGxpYg0KaW1w"
        "b3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHNodXRpbA0KaW1wb3J0IHN1YnByb2Nlc3MN"
        "CmltcG9ydCBzeXMNCmltcG9ydCB0aW1lDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZy"
        "b20gdHlwaW5nIGltcG9ydCBBbnkNCg0KIyBUaGUgZnVsbC1wcmVjaXNpb24gYmFzZSB0aGUg"
        "YWRhcHRlciBpcyBtZXJnZWQgaW50by4gVGhlIGFkYXB0ZXIgd2FzIHRyYWluZWQNCiMgYWdh"
        "aW5zdCB0aGUgYm5iLTRiaXQgdmFyaWFudCBvZiB0aGlzIGV4YWN0IGNoZWNrcG9pbnQ7IG1l"
        "cmdpbmcgaW50byB0aGUNCiMgZnAxNiB3ZWlnaHRzIGlzIHRoZSBzdGFuZGFyZCBRTG9SQSBl"
        "eHBvcnQgcGF0aCBhbmQgdGhlIG1hbmlmZXN0IHJlY29yZHMgYm90aC4NCkRFRkFVTFRfRlVM"
        "TF9CQVNFID0gIlF3ZW4vUXdlbjMtNEItSW5zdHJ1Y3QtMjUwNyINCg0KTU9ERUxGSUxFID0g"
        "IiIiIyBPbGxhbWEgTW9kZWxmaWxlIGZvciBBVFJBLTRCLg0KIw0KIyBUaGUgY2hhdCB0ZW1w"
        "bGF0ZSB0cmF2ZWxzIGluc2lkZSB0aGUgR0dVRiAodGhlIGNvbnZlcnRlciBlbWJlZHMgdGhl"
        "IGJhc2UNCiMgbW9kZWwncyB0b2tlbml6ZXIgY2hhdCB0ZW1wbGF0ZSksIHNvIG5vbmUgaXMg"
        "cmVwZWF0ZWQgaGVyZS4gVGhlIEFUUkEgcnVudGltZQ0KIyBzdXBwbGllcyBpdHMgb3duIHN5"
        "c3RlbSBwcm9tcHQgcGVyIGFnZW50OyBkbyBub3QgYWRkIG9uZS4NCkZST00gLi97Z2d1Zn0N"
        "ClBBUkFNRVRFUiB0ZW1wZXJhdHVyZSAwLjENClBBUkFNRVRFUiBudW1fY3R4IHtjdHh9DQoi"
        "IiINCg0KDQpkZWYgc2hhMjU2X2ZpbGUocGF0aDogUGF0aCkgLT4gc3RyOg0KICAgIGRpZ2Vz"
        "dCA9IGhhc2hsaWIuc2hhMjU2KCkNCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBoYW5k"
        "bGU6DQogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogaGFuZGxlLnJlYWQoMSA8"
        "PCAyMCksIGIiIik6DQogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGNodW5rKQ0KICAgIHJl"
        "dHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkNCg0KDQpkZWYgbG9hZF90cmFpbmluZ19tYW5pZmVz"
        "dChhZGFwdGVyOiBQYXRoKSAtPiBkaWN0W3N0ciwgQW55XToNCiAgICBtYW5pZmVzdF9wYXRo"
        "ID0gYWRhcHRlciAvICJtYW5pZmVzdC5qc29uIg0KICAgIGlmIG5vdCBtYW5pZmVzdF9wYXRo"
        "LmV4aXN0cygpOg0KICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYie21hbmlmZXN0X3BhdGh9"
        "IGlzIG1pc3Npbmc7IGV4cG9ydCBvbmx5IHdoYXQgdHJhaW4ucHkgcHJvZHVjZWQiKQ0KICAg"
        "IG1hbmlmZXN0ID0ganNvbi5sb2FkcyhtYW5pZmVzdF9wYXRoLnJlYWRfdGV4dChlbmNvZGlu"
        "Zz0idXRmLTgiKSkNCiAgICBzdGF0dXMgPSBtYW5pZmVzdC5nZXQoInN0YXR1cyIpDQogICAg"
        "aWYgc3RhdHVzICE9ICJjb21wbGV0ZWQiOg0KICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KA0K"
        "ICAgICAgICAgICAgZiJhZGFwdGVyIG1hbmlmZXN0IHN0YXR1cyBpcyB7c3RhdHVzIXJ9LCBu"
        "b3QgJ2NvbXBsZXRlZCc7ICINCiAgICAgICAgICAgICJhIHNtb2tlIG9yIGZhaWxlZCBydW4g"
        "bXVzdCBub3QgYmUgZXhwb3J0ZWQgYXMgYSBtb2RlbCINCiAgICAgICAgKQ0KICAgIHJldHVy"
        "biBtYW5pZmVzdA0KDQoNCmRlZiBtZXJnZShhZGFwdGVyOiBQYXRoLCBiYXNlOiBzdHIsIG91"
        "dDogUGF0aCkgLT4gUGF0aDoNCiAgICB0cnk6DQogICAgICAgIGltcG9ydCB0b3JjaA0KICAg"
        "ICAgICBmcm9tIHBlZnQgaW1wb3J0IFBlZnRNb2RlbA0KICAgICAgICBmcm9tIHRyYW5zZm9y"
        "bWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEF1dG9Ub2tlbml6ZXINCiAgICBl"
        "eGNlcHQgSW1wb3J0RXJyb3IgYXMgZXJyb3I6ICAjIHByYWdtYTogbm8gY292ZXIgLSBkZXBl"
        "bmRzIG9uIHRoZSBlbnZpcm9ubWVudA0KICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYibWVy"
        "Z2UgZGVwZW5kZW5jaWVzIGFyZSBub3QgaW5zdGFsbGVkICh7ZXJyb3J9KSIpIGZyb20gZXJy"
        "b3INCg0KICAgIG1lcmdlZF9kaXIgPSBvdXQgLyAibWVyZ2VkIg0KICAgIGlmIG1lcmdlZF9k"
        "aXIuZXhpc3RzKCk6DQogICAgICAgIHNodXRpbC5ybXRyZWUobWVyZ2VkX2RpcikNCiAgICBt"
        "ZXJnZWRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkNCg0KICAgIHVzZV9jdWRhID0gdG9yY2gu"
        "Y3VkYS5pc19hdmFpbGFibGUoKQ0KICAgICMgTmF0aXZlIGJmMTYgb25seSAoQW1wZXJlIGFu"
        "ZCBuZXdlcik7IHNlZSBzdXBwb3J0c19iZjE2IGluIHRyYWluLnB5IGZvcg0KICAgICMgd2h5"
        "IHRvcmNoLmN1ZGEuaXNfYmYxNl9zdXBwb3J0ZWQoKSBpcyB0aGUgd3JvbmcgcXVlc3Rpb24g"
        "b24gYSBUNC4NCiAgICBjYXBhYmlsaXR5ID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX2NhcGFi"
        "aWxpdHkoKSBpZiB1c2VfY3VkYSBlbHNlICgwLCAwKQ0KICAgIGR0eXBlID0gdG9yY2guYmZs"
        "b2F0MTYgaWYgY2FwYWJpbGl0eVswXSA+PSA4IGVsc2UgdG9yY2guZmxvYXQxNg0KICAgIHBy"
        "aW50KGYiYmFzZSBtb2RlbCA6IHtiYXNlfSIpDQogICAgcHJpbnQoZiJhZGFwdGVyICAgIDog"
        "e2FkYXB0ZXJ9IikNCiAgICBwcmludChmImR0eXBlICAgICAgOiB7ZHR5cGV9IikNCg0KICAg"
        "IG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKA0KICAgICAg"
        "ICBiYXNlLCB0b3JjaF9kdHlwZT1kdHlwZSwgZGV2aWNlX21hcD0iYXV0byIgaWYgdXNlX2N1"
        "ZGEgZWxzZSBOb25lLCBsb3dfY3B1X21lbV91c2FnZT1UcnVlDQogICAgKQ0KICAgIG1vZGVs"
        "ID0gUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChtb2RlbCwgc3RyKGFkYXB0ZXIpKQ0KICAg"
        "IG1vZGVsID0gbW9kZWwubWVyZ2VfYW5kX3VubG9hZCgpDQogICAgbW9kZWwuc2F2ZV9wcmV0"
        "cmFpbmVkKHN0cihtZXJnZWRfZGlyKSwgc2FmZV9zZXJpYWxpemF0aW9uPVRydWUpDQoNCiAg"
        "ICB0b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChzdHIoYWRhcHRl"
        "cikpDQogICAgdG9rZW5pemVyLnNhdmVfcHJldHJhaW5lZChzdHIobWVyZ2VkX2RpcikpDQog"
        "ICAgcHJpbnQoZiJtZXJnZWQgICAgIDoge21lcmdlZF9kaXJ9IikNCiAgICByZXR1cm4gbWVy"
        "Z2VkX2Rpcg0KDQoNCmRlZiBjb252ZXJ0X2dndWYobWVyZ2VkOiBQYXRoLCBvdXQ6IFBhdGgs"
        "IGxsYW1hX2NwcDogUGF0aCwgZ2d1Zl90eXBlOiBzdHIpIC0+IFBhdGg6DQogICAgY29udmVy"
        "dGVyID0gbGxhbWFfY3BwIC8gImNvbnZlcnRfaGZfdG9fZ2d1Zi5weSINCiAgICBpZiBub3Qg"
        "Y29udmVydGVyLmV4aXN0cygpOg0KICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYie2NvbnZl"
        "cnRlcn0gbm90IGZvdW5kOyBwYXNzIC0tbGxhbWEtY3BwIHBvaW50aW5nIGF0IGEgbGxhbWEu"
        "Y3BwIGNoZWNrb3V0IikNCg0KICAgIGRpcmVjdCA9IHsiZjE2IjogImYxNiIsICJiZjE2Ijog"
        "ImJmMTYiLCAicThfMCI6ICJxOF8wIn0NCiAgICBpZiBnZ3VmX3R5cGUgaW4gZGlyZWN0Og0K"
        "ICAgICAgICB0YXJnZXQgPSBvdXQgLyBmImF0cmEtNGIte2dndWZfdHlwZX0uZ2d1ZiINCiAg"
        "ICAgICAgcnVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKGNvbnZlcnRlciksIHN0cihtZXJnZWQp"
        "LCAiLS1vdXRmaWxlIiwgc3RyKHRhcmdldCksICItLW91dHR5cGUiLCBkaXJlY3RbZ2d1Zl90"
        "eXBlXV0pDQogICAgICAgIHJldHVybiB0YXJnZXQNCg0KICAgICMgQW55dGhpbmcgZWxzZSBn"
        "b2VzIHRocm91Z2ggbGxhbWEtcXVhbnRpemUgZnJvbSBhbiBmMTYgaW50ZXJtZWRpYXRlLg0K"
        "ICAgIGludGVybWVkaWF0ZSA9IG91dCAvICJhdHJhLTRiLWYxNi5nZ3VmIg0KICAgIHJ1bihb"
        "c3lzLmV4ZWN1dGFibGUsIHN0cihjb252ZXJ0ZXIpLCBzdHIobWVyZ2VkKSwgIi0tb3V0Zmls"
        "ZSIsIHN0cihpbnRlcm1lZGlhdGUpLCAiLS1vdXR0eXBlIiwgImYxNiJdKQ0KICAgIHF1YW50"
        "aXplID0gZmluZF9xdWFudGl6ZShsbGFtYV9jcHApDQogICAgaWYgcXVhbnRpemUgaXMgTm9u"
        "ZToNCiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgNCiAgICAgICAgICAgICJsbGFtYS1xdWFu"
        "dGl6ZSBiaW5hcnkgbm90IGZvdW5kOyBidWlsZCBsbGFtYS5jcHAgKGNtYWtlIC1CIGJ1aWxk"
        "ICYmIGNtYWtlIC0tYnVpbGQgYnVpbGQgIg0KICAgICAgICAgICAgIi0tdGFyZ2V0IGxsYW1h"
        "LXF1YW50aXplKSBvciB1c2UgLS1nZ3VmIHE4XzAsIHdoaWNoIG5lZWRzIG5vIGJpbmFyeSIN"
        "CiAgICAgICAgKQ0KICAgIHRhcmdldCA9IG91dCAvIGYiYXRyYS00Yi17Z2d1Zl90eXBlfS5n"
        "Z3VmIg0KICAgIHJ1bihbc3RyKHF1YW50aXplKSwgc3RyKGludGVybWVkaWF0ZSksIHN0cih0"
        "YXJnZXQpLCBnZ3VmX3R5cGUudXBwZXIoKV0pDQogICAgaW50ZXJtZWRpYXRlLnVubGluayht"
        "aXNzaW5nX29rPVRydWUpDQogICAgcmV0dXJuIHRhcmdldA0KDQoNCmRlZiBmaW5kX3F1YW50"
        "aXplKGxsYW1hX2NwcDogUGF0aCkgLT4gUGF0aCB8IE5vbmU6DQogICAgY2FuZGlkYXRlcyA9"
        "IFsNCiAgICAgICAgbGxhbWFfY3BwIC8gImJ1aWxkIiAvICJiaW4iIC8gImxsYW1hLXF1YW50"
        "aXplIiwNCiAgICAgICAgbGxhbWFfY3BwIC8gImJ1aWxkIiAvICJiaW4iIC8gImxsYW1hLXF1"
        "YW50aXplLmV4ZSIsDQogICAgICAgIGxsYW1hX2NwcCAvICJsbGFtYS1xdWFudGl6ZSIsDQog"
        "ICAgXQ0KICAgIGZvciBjYW5kaWRhdGUgaW4gY2FuZGlkYXRlczoNCiAgICAgICAgaWYgY2Fu"
        "ZGlkYXRlLmV4aXN0cygpOg0KICAgICAgICAgICAgcmV0dXJuIGNhbmRpZGF0ZQ0KICAgIGZv"
        "dW5kID0gc2h1dGlsLndoaWNoKCJsbGFtYS1xdWFudGl6ZSIpDQogICAgcmV0dXJuIFBhdGgo"
        "Zm91bmQpIGlmIGZvdW5kIGVsc2UgTm9uZQ0KDQoNCmRlZiBydW4oY29tbWFuZDogbGlzdFtz"
        "dHJdKSAtPiBOb25lOg0KICAgIHByaW50KCIkIiwgIiAiLmpvaW4oY29tbWFuZCkpDQogICAg"
        "c3VicHJvY2Vzcy5ydW4oY29tbWFuZCwgY2hlY2s9VHJ1ZSkNCg0KDQpkZWYgbWFpbigpIC0+"
        "IGludDoNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlv"
        "bj0iTWVyZ2UgYW5kIGV4cG9ydCBBVFJBLTRCIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50"
        "KCItLWFkYXB0ZXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9InRyYWluLnB5"
        "IG91dHB1dCBkaXJlY3RvcnkiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0Iiwg"
        "dHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoImV4cG9ydCIpKQ0KICAgIHBhcnNlci5hZGRfYXJn"
        "dW1lbnQoIi0tYmFzZSIsIGRlZmF1bHQ9b3MuZW52aXJvbi5nZXQoIkFUUkFfRlVMTF9CQVNF"
        "IiwgREVGQVVMVF9GVUxMX0JBU0UpKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbGxh"
        "bWEtY3BwIiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImxsYW1hLmNwcCBjaGVj"
        "a291dCBmb3IgR0dVRiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1nZ3VmIiwgZGVm"
        "YXVsdD1Ob25lLCBoZWxwPSJmMTYgfCBiZjE2IHwgcThfMCB8IHE0X2tfbSB8IHE1X2tfbSAu"
        "Li4iKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2tpcC1tZXJnZSIsIGFjdGlvbj0i"
        "c3RvcmVfdHJ1ZSIsIGhlbHA9InJldXNlIDxvdXQ+L21lcmdlZCBmcm9tIGEgcHJldmlvdXMg"
        "cnVuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWN0eCIsIHR5cGU9aW50LCBkZWZh"
        "dWx0PTQwOTYsIGhlbHA9Im51bV9jdHggd3JpdHRlbiBpbnRvIHRoZSBNb2RlbGZpbGUiKQ0K"
        "ICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpDQoNCiAgICB0cmFpbmluZyA9IGxvYWRf"
        "dHJhaW5pbmdfbWFuaWZlc3QoYXJncy5hZGFwdGVyKQ0KICAgIGFyZ3Mub3V0Lm1rZGlyKHBh"
        "cmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCg0KICAgIHN0YXJ0ZWQgPSB0aW1lLnN0cmZ0"
        "aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKQ0KICAgIG1lcmdlZCA9"
        "IGFyZ3Mub3V0IC8gIm1lcmdlZCINCiAgICBpZiBhcmdzLnNraXBfbWVyZ2U6DQogICAgICAg"
        "IGlmIG5vdCBtZXJnZWQuZXhpc3RzKCk6DQogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0"
        "KGYiLS1za2lwLW1lcmdlIGdpdmVuIGJ1dCB7bWVyZ2VkfSBkb2VzIG5vdCBleGlzdCIpDQog"
        "ICAgZWxzZToNCiAgICAgICAgbWVyZ2VkID0gbWVyZ2UoYXJncy5hZGFwdGVyLCBhcmdzLmJh"
        "c2UsIGFyZ3Mub3V0KQ0KDQogICAgZ2d1Zl9wYXRoOiBQYXRoIHwgTm9uZSA9IE5vbmUNCiAg"
        "ICBpZiBhcmdzLmdndWY6DQogICAgICAgIGlmIGFyZ3MubGxhbWFfY3BwIGlzIE5vbmU6DQog"
        "ICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCItLWdndWYgbmVlZHMgLS1sbGFtYS1jcHAg"
        "PHBhdGggdG8gbGxhbWEuY3BwIGNoZWNrb3V0PiIpDQogICAgICAgIGdndWZfcGF0aCA9IGNv"
        "bnZlcnRfZ2d1ZihtZXJnZWQsIGFyZ3Mub3V0LCBhcmdzLmxsYW1hX2NwcCwgYXJncy5nZ3Vm"
        "Lmxvd2VyKCkpDQogICAgICAgIChhcmdzLm91dCAvICJNb2RlbGZpbGUiKS53cml0ZV90ZXh0"
        "KA0KICAgICAgICAgICAgTU9ERUxGSUxFLmZvcm1hdChnZ3VmPWdndWZfcGF0aC5uYW1lLCBj"
        "dHg9YXJncy5jdHgpLCBlbmNvZGluZz0idXRmLTgiDQogICAgICAgICkNCg0KICAgIG1hbmlm"
        "ZXN0ID0gew0KICAgICAgICAiZXhwb3J0ZWRfYXQiOiBzdGFydGVkLA0KICAgICAgICAiYWRh"
        "cHRlciI6IHN0cihhcmdzLmFkYXB0ZXIpLA0KICAgICAgICAidHJhaW5pbmdfbWFuaWZlc3Qi"
        "OiB0cmFpbmluZywNCiAgICAgICAgImZ1bGxfYmFzZSI6IGFyZ3MuYmFzZSwNCiAgICAgICAg"
        "Im1lcmdlZF9kaXIiOiBzdHIobWVyZ2VkKSwNCiAgICAgICAgImdndWYiOiBOb25lDQogICAg"
        "ICAgIGlmIGdndWZfcGF0aCBpcyBOb25lDQogICAgICAgIGVsc2UgeyJmaWxlIjogZ2d1Zl9w"
        "YXRoLm5hbWUsICJ0eXBlIjogYXJncy5nZ3VmLmxvd2VyKCksICJzaGEyNTYiOiBzaGEyNTZf"
        "ZmlsZShnZ3VmX3BhdGgpLCAiYnl0ZXMiOiBnZ3VmX3BhdGguc3RhdCgpLnN0X3NpemV9LA0K"
        "ICAgICAgICAibm90ZSI6ICgNCiAgICAgICAgICAgICJXZWlnaHRzIGFyZSB0aGUgYmFzZSBt"
        "b2RlbCBwbHVzIHRoZSBhZGFwdGVyIG5hbWVkIGFib3ZlLiBXaGV0aGVyIHRoaXMgZXhwb3J0"
        "ICINCiAgICAgICAgICAgICJtYXkgYmUgY2FsbGVkIEFUUkEtNEIgaXMgZGVjaWRlZCBieSBl"
        "dmFsdWF0ZS5weSBhZ2FpbnN0IHRoZSB0aHJlc2hvbGRzIGluICINCiAgICAgICAgICAgICJj"
        "b25maWcvZGVmYXVsdC55YW1sLCBub3QgYnkgdGhlIGZhY3QgdGhhdCBpdCBleGlzdHMuIg0K"
        "ICAgICAgICApLA0KICAgIH0NCiAgICAoYXJncy5vdXQgLyAiZXhwb3J0LW1hbmlmZXN0Lmpz"
        "b24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKSwgZW5jb2Rp"
        "bmc9InV0Zi04IikNCiAgICBwcmludChmIlxuZXhwb3J0IG1hbmlmZXN0OiB7YXJncy5vdXQg"
        "LyAnZXhwb3J0LW1hbmlmZXN0Lmpzb24nfSIpDQogICAgaWYgZ2d1Zl9wYXRoIGlzIG5vdCBO"
        "b25lOg0KICAgICAgICBwcmludChmImdndWYgICAgICAgICAgIDoge2dndWZfcGF0aH0gKHtt"
        "YW5pZmVzdFsnZ2d1ZiddWydieXRlcyddIC8gKDEwMjQqKjMpOi4yZn0gR2lCKSIpDQogICAg"
        "cmV0dXJuIDANCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5"
        "c3RlbUV4aXQobWFpbigpKQ0K"
    ),
    "config/default.yaml": (
        "IyBBVFJBLTRCIHRyYWluaW5nIGNvbmZpZ3VyYXRpb24uCiMKIyBFdmVyeSBoeXBlci1wYXJh"
        "bWV0ZXIgbGl2ZXMgaGVyZSBzbyBhIHJ1biBpcyByZXByb2R1Y2libGUgZnJvbSBvbmUgZmls"
        "ZSBwbHVzIGEKIyBkYXRhc2V0IGhhc2guIEFueXRoaW5nIGFic2VudCBmcm9tIHRoaXMgZmls"
        "ZSBpcyBhIGRlZmF1bHQgaW4gdHJhaW4ucHkgYW5kIGlzCiMgcmVjb3JkZWQgaW4gdGhlIHJ1"
        "biBtYW5pZmVzdCBhbnl3YXkuCgpydW46CiAgbmFtZTogYXRyYS00Yi12MAogIHNlZWQ6IDQy"
        "CiAgIyBXcml0dGVuIGludG8gdGhlIG1hbmlmZXN0IHNvIGEgY2hlY2twb2ludCBjYW4gYWx3"
        "YXlzIGJlIHRyYWNlZCBiYWNrLgogIG5vdGVzOiAiZmlyc3QgQVRSQS00QiBTRlQgYXR0ZW1w"
        "dCIKCm1vZGVsOgogICMgT3ZlcnJpZGFibGUgd2l0aCBBVFJBX0JBU0UuIFRoZSA0LWJpdCB2"
        "YXJpYW50IGlzIG5hbWVkIGV4cGxpY2l0bHk6IGxldHRpbmcKICAjIHRoZSBsb2FkZXIgcGlj"
        "ayBpdHMgb3duIHF1YW50aXNhdGlvbiBzaWxlbnRseSBjaGFuZ2VzIG1lbW9yeSB1c2UuCiAg"
        "YmFzZTogdW5zbG90aC9Rd2VuMy00Qi1JbnN0cnVjdC0yNTA3LWJuYi00Yml0CiAgbWF4X3Nl"
        "cV9sZW5ndGg6IDIwNDgKICBsb2FkX2luXzRiaXQ6IHRydWUKICAjIGZwMTYgb24gYSBUNCAo"
        "bm8gYmYxNik7IGJmMTYgb24gQW1wZXJlIGFuZCBuZXdlci4gImF1dG8iIGRldGVjdHMuCiAg"
        "cHJlY2lzaW9uOiBhdXRvCgpsb3JhOgogIHI6IDE2CiAgYWxwaGE6IDE2CiAgZHJvcG91dDog"
        "MC4wCiAgdGFyZ2V0X21vZHVsZXM6CiAgICAtIHFfcHJvagogICAgLSBrX3Byb2oKICAgIC0g"
        "dl9wcm9qCiAgICAtIG9fcHJvagogICAgLSBnYXRlX3Byb2oKICAgIC0gdXBfcHJvagogICAg"
        "LSBkb3duX3Byb2oKICAjIFVuc2xvdGgncyBjaGVja3BvaW50aW5nIHZhcmlhbnQ7IGZhbGxz"
        "IGJhY2sgdG8gdGhlIHN0YW5kYXJkIG9uZSBlbHNld2hlcmUuCiAgZ3JhZGllbnRfY2hlY2tw"
        "b2ludGluZzogdW5zbG90aAoKdHJhaW5pbmc6CiAgZXBvY2hzOiAyCiAgcGVyX2RldmljZV9i"
        "YXRjaF9zaXplOiAxCiAgZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzOiA4CiAgbGVhcm5p"
        "bmdfcmF0ZTogMC4wMDAyCiAgbHJfc2NoZWR1bGVyOiBjb3NpbmUKICB3YXJtdXBfcmF0aW86"
        "IDAuMDMKICB3ZWlnaHRfZGVjYXk6IDAuMDEKICBvcHRpbWl6ZXI6IHBhZ2VkX2FkYW13Xzhi"
        "aXQKICBtYXhfZ3JhZF9ub3JtOiAxLjAKICBsb2dnaW5nX3N0ZXBzOiA1CiAgc2F2ZV9zdGVw"
        "czogNTAKICBzYXZlX3RvdGFsX2xpbWl0OiAyCiAgIyBUcmFpbmluZyBvbiBjb21wbGV0aW9u"
        "cyBvbmx5OiB0aGUgbG9zcyBpZ25vcmVzIHRoZSBwcm9tcHQsIHNvIHRoZSBtb2RlbAogICMg"
        "bGVhcm5zIHRvIHByb2R1Y2UgYW5zd2VycyByYXRoZXIgdGhhbiB0byBwcmVkaWN0IHRoZSBl"
        "dmlkZW5jZSBpdCB3YXMgZ2l2ZW4uCiAgdHJhaW5fb25fY29tcGxldGlvbnNfb25seTogdHJ1"
        "ZQoKZGF0YToKICBkaXJlY3Rvcnk6IGRhdGEvb3V0CiAgdHJhaW5fZmlsZTogdHJhaW4uanNv"
        "bmwKICB2YWxpZGF0aW9uX2ZpbGU6IHZhbGlkYXRpb24uanNvbmwKICB0ZXN0X2ZpbGU6IHRl"
        "c3QuanNvbmwKICAjIFRoZSBidWlsZCByZWZ1c2VzIHRvIHByb2NlZWQgYmVsb3cgdGhpcywg"
        "YmVjYXVzZSBhIG1vZGVsIHRoYXQgaGFzIHJhcmVseQogICMgc2VlbiBhIHJlZnVzYWwgd2ls"
        "bCBub3QgcHJvZHVjZSBvbmUgd2hlbiBpdCBtYXR0ZXJzLgogIG1pbl9ub19hY3Rpb25fcmF0"
        "aW86IDAuNDAKCmh1YjoKICAjIFNldCBBVFJBX1BVU0g9MSBhbmQgSEZfVE9LRU4gdG8gZW5h"
        "YmxlLiBPZmYgYnkgZGVmYXVsdCBzbyBhIHJ1biBuZXZlcgogICMgdXBsb2FkcyBhbnl0aGlu"
        "ZyB0aGUgb3BlcmF0b3IgZGlkIG5vdCBhc2sgaXQgdG8uCiAgcHVzaDogZmFsc2UKICByZXBv"
        "X2lkOiAiIgogIHByaXZhdGU6IHRydWUKCmV2YWx1YXRpb246CiAgIyBUaHJlc2hvbGRzIGEg"
        "Y2hlY2twb2ludCBtdXN0IGNsZWFyIGJlZm9yZSBpdCBtYXkgYmUgY2FsbGVkIEFUUkEtNEIg"
        "djAuMS4KICAjIFRoZXkgYXJlIGRlbGliZXJhdGVseSBhYm91dCBjb3JyZWN0bmVzcyBvZiBm"
        "b3JtIGFuZCByZWZ1c2FsLCBub3QgcHJvZml0LgogIHRocmVzaG9sZHM6CiAgICBzdHJ1Y3R1"
        "cmVkX291dHB1dF92YWxpZGl0eTogMC45OAogICAgdG9vbF9zZWxlY3Rpb25fYWNjdXJhY3k6"
        "IDAuODUKICAgIHRvb2xfYXJndW1lbnRfdmFsaWRpdHk6IDAuOTUKICAgIHN0YWxlX2RhdGFf"
        "cmVqZWN0aW9uOiAwLjkwCiAgICBoYWxsdWNpbmF0ZWRfcHJpY2VfcmF0ZV9tYXg6IDAuMDIK"
        "ICAgIG5vX2FjdGlvbl9jb3JyZWN0bmVzczogMC44NQogICAgdW5zdXBwb3J0ZWRfY2hhaW5f"
        "cmVqZWN0aW9uOiAwLjk1CiAgICBscF9hY3Rpb25fdmFsaWRpdHk6IDAuOTAK"
    ),
    "data/__init__.py": (
        ""
    ),
    "data/schema.py": (
        "IiIiVHJhaW5pbmctZXhhbXBsZSBzY2hlbWEgYW5kIHZhbGlkYXRvciBmb3IgQVRSQS00Qi4N"
        "Cg0KRXZlcnkgZXhhbXBsZSBpcyBhIGNvbnZlcnNhdGlvbiB0aGF0IGVuZHMgaW4gYSBzdHJ1"
        "Y3R1cmVkIGRlY2lzaW9uLiBUd28gcnVsZXMNCnNoYXBlIHRoZSB3aG9sZSBzY2hlbWEsIGFu"
        "ZCB0aGUgdmFsaWRhdG9yIGVuZm9yY2VzIGJvdGg6DQoNCjEuICoqTm90aGluZyBpbiB0aGUg"
        "cHJvbXB0IG1heSBwb3N0ZGF0ZSB0aGUgZGVjaXNpb24uKiogQSBoaXN0b3JpY2FsIGV4YW1w"
        "bGUNCiAgIHRoYXQgbGVha3MgdG9tb3Jyb3cncyBwcmljZSB0ZWFjaGVzIHRoZSBtb2RlbCB0"
        "byBleHBlY3QgaW5mb3JtYXRpb24gaXQgd2lsbA0KICAgbmV2ZXIgaGF2ZSBhdCBpbmZlcmVu"
        "Y2UgdGltZSwgYW5kIHRoZSByZXN1bHRpbmcgZXZhbHVhdGlvbiBudW1iZXJzIGFyZQ0KICAg"
        "ZmljdGlvbi4gVGhlIG91dGNvbWUgbGl2ZXMgaW4gYSBzZXBhcmF0ZSBmaWVsZCB0aGF0IGlz"
        "IG5ldmVyIHJlbmRlcmVkLg0KDQoyLiAqKlJlZnVzYWwgaXMgYSBmaXJzdC1jbGFzcyBsYWJl"
        "bC4qKiBgYE5PX0FDVElPTmBgIGFuZCBgYElOU1VGRklDSUVOVF9EQVRBYGANCiAgIGFyZSBj"
        "b3JyZWN0IGFuc3dlcnMsIG5vdCBmYWlsdXJlcywgYW5kIHRoZSBkYXRhc2V0IGlzIHJlcXVp"
        "cmVkIHRvIGNvbnRhaW4NCiAgIGVub3VnaCBvZiB0aGVtIHRoYXQgdGhlIG1vZGVsIGxlYXJu"
        "cyB0byBwcm9kdWNlIHRoZW0uDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5v"
        "dGF0aW9ucw0KDQppbXBvcnQganNvbg0KaW1wb3J0IHJlDQpmcm9tIGRhdGFjbGFzc2VzIGlt"
        "cG9ydCBkYXRhY2xhc3MsIGZpZWxkLCBhc2RpY3QNCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRh"
        "dGV0aW1lLCB0aW1lem9uZQ0KZnJvbSBlbnVtIGltcG9ydCBFbnVtDQpmcm9tIHR5cGluZyBp"
        "bXBvcnQgQW55LCBJdGVyYWJsZQ0KDQoNClNDSEVNQV9WRVJTSU9OID0gMQ0KDQpDSEFJTlMg"
        "PSAoImJhc2UiLCAiYnNjIiwgInJvYmluaG9vZCIsICJzb2xhbmEiKQ0KDQoNCmNsYXNzIERv"
        "bWFpbihzdHIsIEVudW0pOg0KICAgICIiIlRoZSBzZXZlbiBkb21haW5zIHRoZSBkYXRhc2V0"
        "IGNvdmVycy4iIiINCg0KICAgIFRPT0xfVVNFID0gInRvb2xfdXNlIg0KICAgIE1BUktFVF9S"
        "RUFTT05JTkcgPSAibWFya2V0X3JlYXNvbmluZyINCiAgICBPTkNIQUlOX1JFQVNPTklORyA9"
        "ICJvbmNoYWluX3JlYXNvbmluZyINCiAgICBUUkFESU5HX0RFQ0lTSU9OID0gInRyYWRpbmdf"
        "ZGVjaXNpb24iDQogICAgTFBfUkVBU09OSU5HID0gImxwX3JlYXNvbmluZyINCiAgICBSSVNL"
        "X1JFQVNPTklORyA9ICJyaXNrX3JlYXNvbmluZyINCiAgICBDSEFJTl9LTk9XTEVER0UgPSAi"
        "Y2hhaW5fa25vd2xlZGdlIg0KDQoNCmNsYXNzIFRyYWRlQWN0aW9uKHN0ciwgRW51bSk6DQog"
        "ICAgTk9fQUNUSU9OID0gIk5PX0FDVElPTiINCiAgICBPUEVOID0gIk9QRU4iDQogICAgUkVE"
        "VUNFID0gIlJFRFVDRSINCiAgICBDTE9TRSA9ICJDTE9TRSINCiAgICBTV0FQID0gIlNXQVAi"
        "DQoNCg0KY2xhc3MgTHBBY3Rpb24oc3RyLCBFbnVtKToNCiAgICBIT0xEID0gIkhPTEQiDQog"
        "ICAgQUREX0xJUVVJRElUWSA9ICJBRERfTElRVUlESVRZIg0KICAgIFJFTU9WRV9MSVFVSURJ"
        "VFkgPSAiUkVNT1ZFX0xJUVVJRElUWSINCiAgICBSRUJBTEFOQ0UgPSAiUkVCQUxBTkNFIg0K"
        "ICAgIENPTExFQ1RfRkVFUyA9ICJDT0xMRUNUX0ZFRVMiDQogICAgRVhJVCA9ICJFWElUIg0K"
        "DQoNCmNsYXNzIFJlc2VhcmNoU3RhdHVzKHN0ciwgRW51bSk6DQogICAgT0sgPSAiT0siDQog"
        "ICAgSU5TVUZGSUNJRU5UX0RBVEEgPSAiSU5TVUZGSUNJRU5UX0RBVEEiDQoNCg0KQGRhdGFj"
        "bGFzcw0KY2xhc3MgTWVzc2FnZToNCiAgICByb2xlOiBzdHIgICMgc3lzdGVtIHwgdXNlciB8"
        "IGFzc2lzdGFudCB8IHRvb2wNCiAgICBjb250ZW50OiBzdHINCiAgICB0b29sX2NhbGxfaWQ6"
        "IHN0ciB8IE5vbmUgPSBOb25lDQogICAgbmFtZTogc3RyIHwgTm9uZSA9IE5vbmUNCg0KDQpA"
        "ZGF0YWNsYXNzDQpjbGFzcyBUb29sU3BlYzoNCiAgICAiIiJBIHRvb2wgdGhlIG1vZGVsIG1h"
        "eSBjYWxsLCBpbiBKU09OLXNjaGVtYSBmb3JtLiIiIg0KDQogICAgbmFtZTogc3RyDQogICAg"
        "ZGVzY3JpcHRpb246IHN0cg0KICAgIHBhcmFtZXRlcnM6IGRpY3Rbc3RyLCBBbnldDQoNCg0K"
        "QGRhdGFjbGFzcw0KY2xhc3MgT3V0Y29tZToNCiAgICAiIiJXaGF0IGFjdHVhbGx5IGhhcHBl"
        "bmVkIGFmdGVyd2FyZHMuDQoNCiAgICBIZWxkIHNlcGFyYXRlbHkgZnJvbSB0aGUgY29udmVy"
        "c2F0aW9uIGFuZCBuZXZlciByZW5kZXJlZCBpbnRvIGEgcHJvbXB0LiBJdA0KICAgIGV4aXN0"
        "cyBzbyBldmFsdWF0aW9uIGNhbiBhc2sgIndhcyB0aGUgcmVmdXNhbCBjb3JyZWN0PyIgd2l0"
        "aG91dCB0aGUgbW9kZWwNCiAgICBoYXZpbmcgYmVlbiBzaG93biB0aGUgYW5zd2VyLg0KICAg"
        "ICIiIg0KDQogICAgcmVhbGl6ZWRfcmV0dXJuX2JwczogaW50IHwgTm9uZSA9IE5vbmUNCiAg"
        "ICB3YXNfY29ycmVjdDogYm9vbCB8IE5vbmUgPSBOb25lDQogICAgbm90ZXM6IHN0ciA9ICIi"
        "DQoNCg0KQGRhdGFjbGFzcw0KY2xhc3MgRXhhbXBsZToNCiAgICBpZDogc3RyDQogICAgZG9t"
        "YWluOiBEb21haW4NCiAgICBzY2hlbWFfdmVyc2lvbjogaW50ID0gU0NIRU1BX1ZFUlNJT04N"
        "Cg0KICAgIGNoYWluOiBzdHIgfCBOb25lID0gTm9uZQ0KICAgICMgVGhlIGluc3RhbnQgdGhl"
        "IGRlY2lzaW9uIGlzIG1hZGUuIE5vdGhpbmcgaW4gYG1lc3NhZ2VzYCBtYXkgYmUgbmV3ZXIu"
        "DQogICAgZGVjaXNpb25fdGltZTogc3RyID0gIiINCiAgICBjcmVhdGVkX2F0OiBzdHIgPSAi"
        "Ig0KDQogICAgbWVzc2FnZXM6IGxpc3RbTWVzc2FnZV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rv"
        "cnk9bGlzdCkNCiAgICB0b29sczogbGlzdFtUb29sU3BlY10gPSBmaWVsZChkZWZhdWx0X2Zh"
        "Y3Rvcnk9bGlzdCkNCg0KICAgICMgVGhlIHN0cnVjdHVyZWQgYW5zd2VyIHRoZSBtb2RlbCBz"
        "aG91bGQgcHJvZHVjZSwgYXMgYSBkaWN0Lg0KICAgIGV4cGVjdGVkX291dHB1dDogZGljdFtz"
        "dHIsIEFueV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkNCg0KICAgIG91dGNvbWU6"
        "IE91dGNvbWUgfCBOb25lID0gTm9uZQ0KDQogICAgcHJvdmVuYW5jZTogc3RyID0gInN5bnRo"
        "ZXRpYyINCiAgICBsaWNlbnNlOiBzdHIgPSAiTUlUIg0KICAgIHF1YWxpdHlfc2NvcmU6IGZs"
        "b2F0ID0gMS4wDQogICAgc3BsaXQ6IHN0ciA9ICJ0cmFpbiIgICMgdHJhaW4gfCB2YWxpZGF0"
        "aW9uIHwgdGVzdA0KDQogICAgIyBFeHBsaWNpdCBiZWhhdmlvdXIgbGFiZWxzLCBlLmcuICJz"
        "dGFsZSIsICJ1bnN1cHBvcnRlZC1jaGFpbiIsICJyZWZ1c2FsIi4NCiAgICAjIEV2YWx1YXRp"
        "b24gc2VsZWN0cyBvbiB0aGVzZSByYXRoZXIgdGhhbiBncmVwcGluZyB0aGUgdGV4dCwgd2hp"
        "Y2ggaXMgaG93IGENCiAgICAjIG1ldHJpYyBlbmRzIHVwIHNjb3JpbmcgInRocmVzaG9sZCIg"
        "YXMgYSBzdGFsZW5lc3MgY2FzZS4NCiAgICB0YWdzOiBsaXN0W3N0cl0gPSBmaWVsZChkZWZh"
        "dWx0X2ZhY3Rvcnk9bGlzdCkNCg0KICAgIGRlZiB0b19qc29uKHNlbGYpIC0+IHN0cjoNCiAg"
        "ICAgICAgcmV0dXJuIGpzb24uZHVtcHMoYXNkaWN0KHNlbGYpLCBkZWZhdWx0PXN0ciwgc29y"
        "dF9rZXlzPVRydWUpDQoNCg0KY2xhc3MgVmFsaWRhdGlvbkVycm9yKEV4Y2VwdGlvbik6DQog"
        "ICAgIiIiUmFpc2VkIGZvciBhbiBleGFtcGxlIHRoYXQgbXVzdCBub3QgZW50ZXIgdGhlIGRh"
        "dGFzZXQuIiIiDQoNCg0KSVNPID0gIiVZLSVtLSVkVCVIOiVNOiVTWiINCg0KIyBBbnkgdGlt"
        "ZXN0YW1wLWxvb2tpbmcgc3RyaW5nIGluc2lkZSBhIHJlbmRlcmVkIG1lc3NhZ2UuDQpUSU1F"
        "U1RBTVBfUEFUVEVSTiA9IHJlLmNvbXBpbGUociJcZHs0fS1cZHsyfS1cZHsyfVRcZHsyfTpc"
        "ZHsyfTpcZHsyfVoiKQ0KDQojIFBocmFzZXMgdGhhdCBnaXZlIGF3YXkgYSBmdXR1cmUgb3V0"
        "Y29tZS4gQSBwcm9tcHQgY29udGFpbmluZyBvbmUgb2YgdGhlc2UgaXMNCiMgdGVhY2hpbmcg"
        "dGhlIG1vZGVsIHRvIHJlYWQgdGhlIGFuc3dlciBvZmYgdGhlIHF1ZXN0aW9uLg0KTEVBS0FH"
        "RV9QSFJBU0VTID0gKA0KICAgICJpbiBoaW5kc2lnaHQiLA0KICAgICJpdCB0dXJuZWQgb3V0"
        "IiwNCiAgICAidGhlIHByaWNlIGxhdGVyIiwNCiAgICAid291bGQgaGF2ZSByZXR1cm5lZCIs"
        "DQogICAgImFzIHdlIG5vdyBrbm93IiwNCiAgICAidGhlIGNvcnJlY3QgYW5zd2VyIGlzIiwN"
        "CiAgICAiZXZlbnR1YWxseSByb3NlIiwNCiAgICAiZXZlbnR1YWxseSBmZWxsIiwNCikNCg0K"
        "DQpkZWYgcGFyc2VfdGltZSh2YWx1ZTogc3RyKSAtPiBkYXRldGltZToNCiAgICByZXR1cm4g"
        "ZGF0ZXRpbWUuc3RycHRpbWUodmFsdWUsIElTTykucmVwbGFjZSh0emluZm89dGltZXpvbmUu"
        "dXRjKQ0KDQoNCmRlZiB2YWxpZGF0ZShleGFtcGxlOiBFeGFtcGxlKSAtPiBOb25lOg0KICAg"
        "ICIiIlJhaXNlIDpjbGFzczpgVmFsaWRhdGlvbkVycm9yYCBpZiB0aGUgZXhhbXBsZSBpcyB1"
        "bnVzYWJsZS4NCg0KICAgIERlbGliZXJhdGVseSBzdHJpY3QuIEEgZGF0YXNldCBpcyB0aGUg"
        "b25lIGFydGVmYWN0IHdob3NlIGRlZmVjdHMgYXJlDQogICAgaW52aXNpYmxlIGluIHRoZSBm"
        "aW5hbCBtb2RlbCwgc28gYW55dGhpbmcgYW1iaWd1b3VzIGlzIHJlamVjdGVkIHJhdGhlciB0"
        "aGFuDQogICAgcmVwYWlyZWQuDQogICAgIiIiDQogICAgaWYgZXhhbXBsZS5zY2hlbWFfdmVy"
        "c2lvbiAhPSBTQ0hFTUFfVkVSU0lPTjoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9y"
        "KGYie2V4YW1wbGUuaWR9OiB1bnN1cHBvcnRlZCBzY2hlbWEgdmVyc2lvbiIpDQoNCiAgICBp"
        "ZiBub3QgZXhhbXBsZS5pZDoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKCJhbiBl"
        "eGFtcGxlIGhhcyBubyBpZCIpDQoNCiAgICBpZiBleGFtcGxlLmNoYWluIGlzIG5vdCBOb25l"
        "IGFuZCBleGFtcGxlLmNoYWluIG5vdCBpbiBDSEFJTlM6DQogICAgICAgIHJhaXNlIFZhbGlk"
        "YXRpb25FcnJvcihmIntleGFtcGxlLmlkfTogdW5zdXBwb3J0ZWQgY2hhaW4ge2V4YW1wbGUu"
        "Y2hhaW4hcn0iKQ0KDQogICAgaWYgbm90IGV4YW1wbGUuZGVjaXNpb25fdGltZToNCiAgICAg"
        "ICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9OiBkZWNpc2lvbl90aW1l"
        "IGlzIHJlcXVpcmVkIikNCg0KICAgIHRyeToNCiAgICAgICAgZGVjaXNpb25fYXQgPSBwYXJz"
        "ZV90aW1lKGV4YW1wbGUuZGVjaXNpb25fdGltZSkNCiAgICBleGNlcHQgVmFsdWVFcnJvciBh"
        "cyBlcnJvcjoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9"
        "OiBtYWxmb3JtZWQgZGVjaXNpb25fdGltZSIpIGZyb20gZXJyb3INCg0KICAgIGlmIG5vdCBl"
        "eGFtcGxlLm1lc3NhZ2VzOg0KICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhh"
        "bXBsZS5pZH06IG5vIG1lc3NhZ2VzIikNCg0KICAgIHJvbGVzID0gW21lc3NhZ2Uucm9sZSBm"
        "b3IgbWVzc2FnZSBpbiBleGFtcGxlLm1lc3NhZ2VzXQ0KICAgIGlmIHJvbGVzWzBdICE9ICJz"
        "eXN0ZW0iOg0KICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06"
        "IHRoZSBmaXJzdCBtZXNzYWdlIG11c3QgYmUgdGhlIHN5c3RlbSBwcm9tcHQiKQ0KICAgIGlm"
        "IHJvbGVzWy0xXSAhPSAiYXNzaXN0YW50IjoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVy"
        "cm9yKGYie2V4YW1wbGUuaWR9OiB0aGUgbGFzdCBtZXNzYWdlIG11c3QgYmUgdGhlIGFzc2lz"
        "dGFudCBhbnN3ZXIiKQ0KICAgIGZvciByb2xlIGluIHJvbGVzOg0KICAgICAgICBpZiByb2xl"
        "IG5vdCBpbiAoInN5c3RlbSIsICJ1c2VyIiwgImFzc2lzdGFudCIsICJ0b29sIik6DQogICAg"
        "ICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06IHVua25vd24g"
        "cm9sZSB7cm9sZSFyfSIpDQoNCiAgICBfY2hlY2tfbm9fZnV0dXJlX2xlYWthZ2UoZXhhbXBs"
        "ZSwgZGVjaXNpb25fYXQpDQogICAgX2NoZWNrX2V4cGVjdGVkX291dHB1dChleGFtcGxlKQ0K"
        "DQogICAgaWYgZXhhbXBsZS5zcGxpdCBub3QgaW4gKCJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwg"
        "InRlc3QiKToNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9"
        "OiB1bmtub3duIHNwbGl0IHtleGFtcGxlLnNwbGl0IXJ9IikNCg0KDQpkZWYgX2NoZWNrX25v"
        "X2Z1dHVyZV9sZWFrYWdlKGV4YW1wbGU6IEV4YW1wbGUsIGRlY2lzaW9uX2F0OiBkYXRldGlt"
        "ZSkgLT4gTm9uZToNCiAgICAiIiJSZWZ1c2UgYW55IHByb21wdCB0aGF0IGNvbnRhaW5zIGlu"
        "Zm9ybWF0aW9uIGZyb20gYWZ0ZXIgdGhlIGRlY2lzaW9uLiIiIg0KICAgIHByb21wdF9tZXNz"
        "YWdlcyA9IFttIGZvciBtIGluIGV4YW1wbGUubWVzc2FnZXMgaWYgbS5yb2xlICE9ICJhc3Np"
        "c3RhbnQiXQ0KDQogICAgZm9yIG1lc3NhZ2UgaW4gcHJvbXB0X21lc3NhZ2VzOg0KICAgICAg"
        "ICBsb3dlcmVkID0gbWVzc2FnZS5jb250ZW50Lmxvd2VyKCkNCiAgICAgICAgZm9yIHBocmFz"
        "ZSBpbiBMRUFLQUdFX1BIUkFTRVM6DQogICAgICAgICAgICBpZiBwaHJhc2UgaW4gbG93ZXJl"
        "ZDoNCiAgICAgICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoDQogICAgICAgICAg"
        "ICAgICAgICAgIGYie2V4YW1wbGUuaWR9OiBwcm9tcHQgY29udGFpbnMgb3V0Y29tZSBsYW5n"
        "dWFnZSAoe3BocmFzZSFyfSkiDQogICAgICAgICAgICAgICAgKQ0KDQogICAgICAgIGZvciBt"
        "YXRjaCBpbiBUSU1FU1RBTVBfUEFUVEVSTi5maW5kYWxsKG1lc3NhZ2UuY29udGVudCk6DQog"
        "ICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgc3RhbXAgPSBwYXJzZV90aW1lKG1h"
        "dGNoKQ0KICAgICAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6DQogICAgICAgICAgICAgICAg"
        "Y29udGludWUNCiAgICAgICAgICAgIGlmIHN0YW1wID4gZGVjaXNpb25fYXQ6DQogICAgICAg"
        "ICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKA0KICAgICAgICAgICAgICAgICAgICBm"
        "IntleGFtcGxlLmlkfTogcHJvbXB0IHJlZmVyZW5jZXMge21hdGNofSwgYWZ0ZXIgZGVjaXNp"
        "b25fdGltZSINCiAgICAgICAgICAgICAgICApDQoNCiAgICAjIFRoZSBvdXRjb21lIG11c3Qg"
        "bmV2ZXIgYXBwZWFyIGluIHRoZSBjb252ZXJzYXRpb24gYXQgYWxsLg0KICAgIGlmIGV4YW1w"
        "bGUub3V0Y29tZSBpcyBub3QgTm9uZSBhbmQgZXhhbXBsZS5vdXRjb21lLm5vdGVzOg0KICAg"
        "ICAgICByZW5kZXJlZCA9ICIgIi5qb2luKG0uY29udGVudCBmb3IgbSBpbiBleGFtcGxlLm1l"
        "c3NhZ2VzKS5sb3dlcigpDQogICAgICAgIGlmIGV4YW1wbGUub3V0Y29tZS5ub3Rlcy5sb3dl"
        "cigpWzo0MF0gaW4gcmVuZGVyZWQ6DQogICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJy"
        "b3IoZiJ7ZXhhbXBsZS5pZH06IHRoZSBvdXRjb21lIG5vdGUgYXBwZWFycyBpbiB0aGUgY29u"
        "dmVyc2F0aW9uIikNCg0KDQpkZWYgX2NoZWNrX2V4cGVjdGVkX291dHB1dChleGFtcGxlOiBF"
        "eGFtcGxlKSAtPiBOb25lOg0KICAgICIiIlRoZSBhbnN3ZXIgbXVzdCBtYXRjaCB0aGUgZG9t"
        "YWluJ3Mgc2NoZW1hIGV4YWN0bHkuIiIiDQogICAgb3V0cHV0ID0gZXhhbXBsZS5leHBlY3Rl"
        "ZF9vdXRwdXQNCiAgICBpZiBub3QgaXNpbnN0YW5jZShvdXRwdXQsIGRpY3QpIG9yIG5vdCBv"
        "dXRwdXQ6DQogICAgICAgIHJhaXNlIFZhbGlkYXRpb25FcnJvcihmIntleGFtcGxlLmlkfTog"
        "ZXhwZWN0ZWRfb3V0cHV0IG11c3QgYmUgYSBub24tZW1wdHkgb2JqZWN0IikNCg0KICAgIGlm"
        "IGV4YW1wbGUuZG9tYWluIGlzIERvbWFpbi5UUkFESU5HX0RFQ0lTSU9OOg0KICAgICAgICBh"
        "Y3Rpb24gPSBvdXRwdXQuZ2V0KCJhY3Rpb24iKQ0KICAgICAgICBpZiBhY3Rpb24gbm90IGlu"
        "IHthLnZhbHVlIGZvciBhIGluIFRyYWRlQWN0aW9ufToNCiAgICAgICAgICAgIHJhaXNlIFZh"
        "bGlkYXRpb25FcnJvcihmIntleGFtcGxlLmlkfTogaW52YWxpZCB0cmFkZSBhY3Rpb24ge2Fj"
        "dGlvbiFyfSIpDQogICAgICAgIGZvciBrZXkgaW4gKCJjaGFpbiIsICJyZWFzb24iLCAiY29u"
        "ZmlkZW5jZSIpOg0KICAgICAgICAgICAgaWYga2V5IG5vdCBpbiBvdXRwdXQ6DQogICAgICAg"
        "ICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9OiB0cmFkaW5n"
        "IG91dHB1dCBpcyBtaXNzaW5nIHtrZXl9IikNCiAgICAgICAgY29uZmlkZW5jZSA9IG91dHB1"
        "dC5nZXQoImNvbmZpZGVuY2UiKQ0KICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShjb25maWRl"
        "bmNlLCAoaW50LCBmbG9hdCkpIG9yIG5vdCAwLjAgPD0gZmxvYXQoY29uZmlkZW5jZSkgPD0g"
        "MS4wOg0KICAgICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9"
        "OiBjb25maWRlbmNlIG11c3QgYmUgYmV0d2VlbiAwIGFuZCAxIikNCiAgICAgICAgIyBBIHJl"
        "ZnVzYWwgdGhhdCBjbGFpbXMgaGlnaCBjb25maWRlbmNlIGlzIGluY29oZXJlbnQgYW5kIHRl"
        "YWNoZXMgdGhlDQogICAgICAgICMgbW9kZWwgdGhhdCB0aGUgZmllbGQgbWVhbnMgbm90aGlu"
        "Zy4NCiAgICAgICAgaWYgYWN0aW9uID09IFRyYWRlQWN0aW9uLk5PX0FDVElPTi52YWx1ZSBh"
        "bmQgZmxvYXQoY29uZmlkZW5jZSkgPiAwLjk6DQogICAgICAgICAgICByYWlzZSBWYWxpZGF0"
        "aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06IE5PX0FDVElPTiB3aXRoIG5lYXItY2VydGFpbiBj"
        "b25maWRlbmNlIikNCg0KICAgIGVsaWYgZXhhbXBsZS5kb21haW4gaXMgRG9tYWluLkxQX1JF"
        "QVNPTklORzoNCiAgICAgICAgYWN0aW9uID0gb3V0cHV0LmdldCgiYWN0aW9uIikNCiAgICAg"
        "ICAgaWYgYWN0aW9uIG5vdCBpbiB7YS52YWx1ZSBmb3IgYSBpbiBMcEFjdGlvbn06DQogICAg"
        "ICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06IGludmFsaWQg"
        "TFAgYWN0aW9uIHthY3Rpb24hcn0iKQ0KDQogICAgZWxpZiBleGFtcGxlLmRvbWFpbiBpcyBE"
        "b21haW4uVE9PTF9VU0U6DQogICAgICAgIGlmICJ0b29sIiBub3QgaW4gb3V0cHV0IGFuZCAi"
        "c3RhdHVzIiBub3QgaW4gb3V0cHV0Og0KICAgICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVy"
        "cm9yKGYie2V4YW1wbGUuaWR9OiB0b29sLXVzZSBvdXRwdXQgbmVlZHMgYSB0b29sIG9yIGEg"
        "c3RhdHVzIikNCiAgICAgICAgdG9vbCA9IG91dHB1dC5nZXQoInRvb2wiKQ0KICAgICAgICBp"
        "ZiB0b29sIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgZGVjbGFyZWQgPSB7c3BlYy5uYW1l"
        "IGZvciBzcGVjIGluIGV4YW1wbGUudG9vbHN9DQogICAgICAgICAgICBpZiB0b29sIG5vdCBp"
        "biBkZWNsYXJlZDoNCiAgICAgICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7"
        "ZXhhbXBsZS5pZH06IGNhbGxzIHVuZGVjbGFyZWQgdG9vbCB7dG9vbCFyfSIpDQoNCiAgICBl"
        "bGlmIGV4YW1wbGUuZG9tYWluIGluIChEb21haW4uTUFSS0VUX1JFQVNPTklORywgRG9tYWlu"
        "Lk9OQ0hBSU5fUkVBU09OSU5HKToNCiAgICAgICAgc3RhdHVzID0gb3V0cHV0LmdldCgic3Rh"
        "dHVzIikNCiAgICAgICAgaWYgc3RhdHVzIG5vdCBpbiB7cy52YWx1ZSBmb3IgcyBpbiBSZXNl"
        "YXJjaFN0YXR1c306DQogICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhh"
        "bXBsZS5pZH06IGludmFsaWQgcmVzZWFyY2ggc3RhdHVzIHtzdGF0dXMhcn0iKQ0KDQoNCmRl"
        "ZiB2YWxpZGF0ZV9hbGwoZXhhbXBsZXM6IEl0ZXJhYmxlW0V4YW1wbGVdKSAtPiBsaXN0W3N0"
        "cl06DQogICAgIiIiVmFsaWRhdGUgYSBjb2xsZWN0aW9uLCByZXR1cm5pbmcgZXZlcnkgZXJy"
        "b3IgcmF0aGVyIHRoYW4gdGhlIGZpcnN0LiIiIg0KICAgIGVycm9yczogbGlzdFtzdHJdID0g"
        "W10NCiAgICBzZWVuX2lkczogc2V0W3N0cl0gPSBzZXQoKQ0KDQogICAgZm9yIGV4YW1wbGUg"
        "aW4gZXhhbXBsZXM6DQogICAgICAgIGlmIGV4YW1wbGUuaWQgaW4gc2Vlbl9pZHM6DQogICAg"
        "ICAgICAgICBlcnJvcnMuYXBwZW5kKGYie2V4YW1wbGUuaWR9OiBkdXBsaWNhdGUgaWQiKQ0K"
        "ICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgc2Vlbl9pZHMuYWRkKGV4YW1wbGUuaWQp"
        "DQoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgdmFsaWRhdGUoZXhhbXBsZSkNCiAgICAg"
        "ICAgZXhjZXB0IFZhbGlkYXRpb25FcnJvciBhcyBlcnJvcjoNCiAgICAgICAgICAgIGVycm9y"
        "cy5hcHBlbmQoc3RyKGVycm9yKSkNCg0KICAgIHJldHVybiBlcnJvcnMNCg=="
    ),
    "data/build.py": (
        "..."
        "U3lzdGVtRXhpdChtYWluKCkpDQo="
    ),
    "data/checks.py": (
        "IiIiRGF0YXNldCBxdWFsaXR5IGdhdGVzLg0KDQpUaGVzZSBydW4gYmVmb3JlIGFueSB0cmFp"
        "bmluZyBhbmQgZmFpbCB0aGUgYnVpbGQgcmF0aGVyIHRoYW4gd2Fybi4gQSBkYXRhc2V0DQpk"
        "ZWZlY3QgaXMgaW52aXNpYmxlIGluIHRoZSB0cmFpbmVkIG1vZGVsIGJ1dCBzaG93cyB1cCBh"
        "cyBhIGNvbmZpZGVudCB3cm9uZw0KYW5zd2VyIG1vbnRocyBsYXRlciwgc28gdGhlIGNoZWNr"
        "cyBhcmUgZGVsaWJlcmF0ZWx5IHVuZm9yZ2l2aW5nLg0KDQpXaGF0IGlzIGNoZWNrZWQ6DQoN"
        "Ci0gKipMZWFrYWdlKiog4oCUIG5vdGhpbmcgaW4gYSBwcm9tcHQgbWF5IHBvc3RkYXRlIGl0"
        "cyBkZWNpc2lvbiB0aW1lLCBhbmQgbm8NCiAgaGlzdG9yaWNhbCBleGFtcGxlIG1heSBhcHBl"
        "YXIgaW4gYSBzcGxpdCBlYXJsaWVyIHRoYW4gb25lIGl0IGNocm9ub2xvZ2ljYWxseQ0KICBm"
        "b2xsb3dzLg0KLSAqKkR1cGxpY2F0aW9uKiog4oCUIGV4YWN0IGFuZCBuZWFyLWR1cGxpY2F0"
        "ZSBwcm9tcHRzIGluZmxhdGUgYXBwYXJlbnQgZGF0YXNldA0KICBzaXplIGFuZCBsZXQgdGhl"
        "IG1vZGVsIG1lbW9yaXNlIGluc3RlYWQgb2YgZ2VuZXJhbGlzZS4NCi0gKipCYWxhbmNlKiog"
        "4oCUIHJlZnVzYWxzIG11c3QgYmUgd2VsbCByZXByZXNlbnRlZC4gQSBtb2RlbCB0aGF0IGhh"
        "cyByYXJlbHkgc2Vlbg0KICBgYE5PX0FDVElPTmBgIHdpbGwgbm90IHByb2R1Y2UgaXQgd2hl"
        "biBpdCBtYXR0ZXJzLg0KLSAqKkNoYWluIGNvdmVyYWdlKiog4oCUIGFsbCBmb3VyIGNoYWlu"
        "cyBtdXN0IGFwcGVhciwgc28gdGhlIG1vZGVsIGRvZXMgbm90IGxlYXJuDQogIHRoYXQgImNo"
        "YWluIiBtZWFucyBCYXNlLg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3Rh"
        "dGlvbnMNCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgaGFzaGxpYg0KaW1wb3J0IGpzb24N"
        "CmltcG9ydCBzeXMNCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXINCmZyb20gZGF0"
        "YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpm"
        "cm9tIHR5cGluZyBpbXBvcnQgSXRlcmFibGUNCg0KZnJvbSAuc2NoZW1hIGltcG9ydCAoDQog"
        "ICAgQ0hBSU5TLA0KICAgIERvbWFpbiwNCiAgICBFeGFtcGxlLA0KICAgIE1lc3NhZ2UsDQog"
        "ICAgT3V0Y29tZSwNCiAgICBUb29sU3BlYywNCiAgICBUcmFkZUFjdGlvbiwNCiAgICBwYXJz"
        "ZV90aW1lLA0KICAgIHZhbGlkYXRlX2FsbCwNCikNCg0KDQojIEEgZGF0YXNldCB3aXRoIGZl"
        "d2VyIHJlZnVzYWxzIHRoYW4gdGhpcyB0ZWFjaGVzIHRoZSBtb2RlbCB0aGF0IGRvaW5nIG5v"
        "dGhpbmcNCiMgaXMgYW4gZWRnZSBjYXNlLiBJdCBpcyB0aGUgc2luZ2xlIG1vc3QgaW1wb3J0"
        "YW50IGJlaGF2aW91ciBBVFJBIG5lZWRzLg0KTUlOX05PX0FDVElPTl9SQVRJTyA9IDAuNDAN"
        "Cg0KIyBOZWFyLWR1cGxpY2F0ZSB0aHJlc2hvbGQgb24gdG9rZW4tc2V0IHNpbWlsYXJpdHku"
        "DQpNQVhfSkFDQ0FSRCA9IDAuOTANCg0KDQpAZGF0YWNsYXNzDQpjbGFzcyBDaGVja1JlcG9y"
        "dDoNCiAgICB0b3RhbDogaW50DQogICAgZXJyb3JzOiBsaXN0W3N0cl0NCiAgICB3YXJuaW5n"
        "czogbGlzdFtzdHJdDQogICAgc3RhdHM6IGRpY3Rbc3RyLCBvYmplY3RdDQoNCiAgICBAcHJv"
        "cGVydHkNCiAgICBkZWYgb2soc2VsZikgLT4gYm9vbDoNCiAgICAgICAgcmV0dXJuIG5vdCBz"
        "ZWxmLmVycm9ycw0KDQogICAgZGVmIHJlbmRlcihzZWxmKSAtPiBzdHI6DQogICAgICAgIGxp"
        "bmVzID0gW2YiZXhhbXBsZXM6IHtzZWxmLnRvdGFsfSJdDQogICAgICAgIGZvciBrZXksIHZh"
        "bHVlIGluIHNvcnRlZChzZWxmLnN0YXRzLml0ZW1zKCkpOg0KICAgICAgICAgICAgbGluZXMu"
        "YXBwZW5kKGYiICB7a2V5fToge3ZhbHVlfSIpDQogICAgICAgIGlmIHNlbGYud2FybmluZ3M6"
        "DQogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZiJ3YXJuaW5ncyAoe2xlbihzZWxmLndhcm5p"
        "bmdzKX0pOiIpDQogICAgICAgICAgICBsaW5lcy5leHRlbmQoZiIgIC0ge3dhcm5pbmd9IiBm"
        "b3Igd2FybmluZyBpbiBzZWxmLndhcm5pbmdzWzoyMF0pDQogICAgICAgIGlmIHNlbGYuZXJy"
        "b3JzOg0KICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYiRVJST1JTICh7bGVuKHNlbGYuZXJy"
        "b3JzKX0pOiIpDQogICAgICAgICAgICBsaW5lcy5leHRlbmQoZiIgIC0ge2Vycm9yfSIgZm9y"
        "IGVycm9yIGluIHNlbGYuZXJyb3JzWzo0MF0pDQogICAgICAgIGVsc2U6DQogICAgICAgICAg"
        "ICBsaW5lcy5hcHBlbmQoIm5vIGVycm9ycyIpDQogICAgICAgIHJldHVybiAiXG4iLmpvaW4o"
        "bGluZXMpDQoNCg0KZGVmIGxvYWRfanNvbmwocGF0aDogUGF0aCkgLT4gbGlzdFtFeGFtcGxl"
        "XToNCiAgICBleGFtcGxlczogbGlzdFtFeGFtcGxlXSA9IFtdDQoNCiAgICBmb3IgbGluZV9u"
        "dW1iZXIsIGxpbmUgaW4gZW51bWVyYXRlKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYt"
        "OCIpLnNwbGl0bGluZXMoKSwgc3RhcnQ9MSk6DQogICAgICAgIGxpbmUgPSBsaW5lLnN0cmlw"
        "KCkNCiAgICAgICAgaWYgbm90IGxpbmU6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAg"
        "ICB0cnk6DQogICAgICAgICAgICByYXcgPSBqc29uLmxvYWRzKGxpbmUpDQogICAgICAgIGV4"
        "Y2VwdCBqc29uLkpTT05EZWNvZGVFcnJvciBhcyBlcnJvcjoNCiAgICAgICAgICAgIHJhaXNl"
        "IFZhbHVlRXJyb3IoZiJ7cGF0aH06e2xpbmVfbnVtYmVyfTogbWFsZm9ybWVkIEpTT04iKSBm"
        "cm9tIGVycm9yDQogICAgICAgIGV4YW1wbGVzLmFwcGVuZChfZnJvbV9kaWN0KHJhdykpDQoN"
        "CiAgICByZXR1cm4gZXhhbXBsZXMNCg0KDQpkZWYgX2Zyb21fZGljdChyYXc6IGRpY3QpIC0+"
        "IEV4YW1wbGU6DQogICAgb3V0Y29tZSA9IHJhdy5nZXQoIm91dGNvbWUiKQ0KICAgIHJldHVy"
        "biBFeGFtcGxlKA0KICAgICAgICBpZD1yYXdbImlkIl0sDQogICAgICAgIGRvbWFpbj1Eb21h"
        "aW4ocmF3WyJkb21haW4iXSksDQogICAgICAgIHNjaGVtYV92ZXJzaW9uPXJhdy5nZXQoInNj"
        "aGVtYV92ZXJzaW9uIiwgMSksDQogICAgICAgIGNoYWluPXJhdy5nZXQoImNoYWluIiksDQog"
        "ICAgICAgIGRlY2lzaW9uX3RpbWU9cmF3LmdldCgiZGVjaXNpb25fdGltZSIsICIiKSwNCiAg"
        "ICAgICAgY3JlYXRlZF9hdD1yYXcuZ2V0KCJjcmVhdGVkX2F0IiwgIiIpLA0KICAgICAgICBt"
        "ZXNzYWdlcz1bTWVzc2FnZSgqKm1lc3NhZ2UpIGZvciBtZXNzYWdlIGluIHJhdy5nZXQoIm1l"
        "c3NhZ2VzIiwgW10pXSwNCiAgICAgICAgdG9vbHM9W1Rvb2xTcGVjKCoqdG9vbCkgZm9yIHRv"
        "b2wgaW4gcmF3LmdldCgidG9vbHMiLCBbXSldLA0KICAgICAgICBleHBlY3RlZF9vdXRwdXQ9"
        "cmF3LmdldCgiZXhwZWN0ZWRfb3V0cHV0Iiwge30pLA0KICAgICAgICBvdXRjb21lPU91dGNv"
        "bWUoKipvdXRjb21lKSBpZiBvdXRjb21lIGVsc2UgTm9uZSwNCiAgICAgICAgcHJvdmVuYW5j"
        "ZT1yYXcuZ2V0KCJwcm92ZW5hbmNlIiwgInN5bnRoZXRpYyIpLA0KICAgICAgICBsaWNlbnNl"
        "PXJhdy5nZXQoImxpY2Vuc2UiLCAiTUlUIiksDQogICAgICAgIHF1YWxpdHlfc2NvcmU9cmF3"
        "LmdldCgicXVhbGl0eV9zY29yZSIsIDEuMCksDQogICAgICAgIHNwbGl0PXJhdy5nZXQoInNw"
        "bGl0IiwgInRyYWluIiksDQogICAgICAgIHRhZ3M9bGlzdChyYXcuZ2V0KCJ0YWdzIiwgW10p"
        "KSwNCiAgICApDQoNCg0KZGVmIGNoZWNrKGV4YW1wbGVzOiBsaXN0W0V4YW1wbGVdKSAtPiBD"
        "aGVja1JlcG9ydDoNCiAgICBlcnJvcnMgPSB2YWxpZGF0ZV9hbGwoZXhhbXBsZXMpDQogICAg"
        "d2FybmluZ3M6IGxpc3Rbc3RyXSA9IFtdDQoNCiAgICBlcnJvcnMuZXh0ZW5kKF9jaGVja19k"
        "dXBsaWNhdGVzKGV4YW1wbGVzKSkNCiAgICBlcnJvcnMuZXh0ZW5kKF9jaGVja19jaHJvbm9s"
        "b2dpY2FsX3NwbGl0cyhleGFtcGxlcykpDQogICAgd2FybmluZ3MuZXh0ZW5kKF9jaGVja19u"
        "ZWFyX2R1cGxpY2F0ZXMoZXhhbXBsZXMpKQ0KDQogICAgc3RhdHMgPSBfc3RhdGlzdGljcyhl"
        "eGFtcGxlcykNCg0KICAgIHJlZnVzYWxfcmF0aW8gPSBzdGF0cy5nZXQoIm5vX2FjdGlvbl9y"
        "YXRpbyIsIDAuMCkNCiAgICBpZiBpc2luc3RhbmNlKHJlZnVzYWxfcmF0aW8sIGZsb2F0KSBh"
        "bmQgc3RhdHMuZ2V0KCJ0cmFkaW5nX2V4YW1wbGVzIiwgMCk6DQogICAgICAgIGlmIHJlZnVz"
        "YWxfcmF0aW8gPCBNSU5fTk9fQUNUSU9OX1JBVElPOg0KICAgICAgICAgICAgZXJyb3JzLmFw"
        "cGVuZCgNCiAgICAgICAgICAgICAgICBmIm9ubHkge3JlZnVzYWxfcmF0aW86LjAlfSBvZiB0"
        "cmFkaW5nIGV4YW1wbGVzIGFyZSBOT19BQ1RJT047ICINCiAgICAgICAgICAgICAgICBmImF0"
        "IGxlYXN0IHtNSU5fTk9fQUNUSU9OX1JBVElPOi4wJX0gaXMgcmVxdWlyZWQiDQogICAgICAg"
        "ICAgICApDQoNCiAgICBtaXNzaW5nX2NoYWlucyA9IFtjaGFpbiBmb3IgY2hhaW4gaW4gQ0hB"
        "SU5TIGlmIHN0YXRzLmdldChmImNoYWluX3tjaGFpbn0iLCAwKSA9PSAwXQ0KICAgIGlmIG1p"
        "c3NpbmdfY2hhaW5zOg0KICAgICAgICBlcnJvcnMuYXBwZW5kKGYibm8gZXhhbXBsZXMgZm9y"
        "IGNoYWluKHMpOiB7JywgJy5qb2luKG1pc3NpbmdfY2hhaW5zKX0iKQ0KDQogICAgZm9yIHNw"
        "bGl0IGluICgidHJhaW4iLCAidmFsaWRhdGlvbiIsICJ0ZXN0Iik6DQogICAgICAgIGlmIHN0"
        "YXRzLmdldChmInNwbGl0X3tzcGxpdH0iLCAwKSA9PSAwOg0KICAgICAgICAgICAgZXJyb3Jz"
        "LmFwcGVuZChmInNwbGl0IHtzcGxpdCFyfSBpcyBlbXB0eSIpDQoNCiAgICByZXR1cm4gQ2hl"
        "Y2tSZXBvcnQodG90YWw9bGVuKGV4YW1wbGVzKSwgZXJyb3JzPWVycm9ycywgd2FybmluZ3M9"
        "d2FybmluZ3MsIHN0YXRzPXN0YXRzKQ0KDQoNCmRlZiBfY2hlY2tfZHVwbGljYXRlcyhleGFt"
        "cGxlczogSXRlcmFibGVbRXhhbXBsZV0pIC0+IGxpc3Rbc3RyXToNCiAgICAiIiJFeGFjdCBk"
        "dXBsaWNhdGUgcHJvbXB0cywgYnkgaGFzaCBvZiB0aGUgbm9uLWFzc2lzdGFudCB0dXJucy4i"
        "IiINCiAgICBzZWVuOiBkaWN0W3N0ciwgc3RyXSA9IHt9DQogICAgZXJyb3JzOiBsaXN0W3N0"
        "cl0gPSBbXQ0KDQogICAgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXM6DQogICAgICAgIGRpZ2Vz"
        "dCA9IF9wcm9tcHRfaGFzaChleGFtcGxlKQ0KICAgICAgICBpZiBkaWdlc3QgaW4gc2VlbjoN"
        "CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7ZXhhbXBsZS5pZH06IGR1cGxpY2F0ZSBw"
        "cm9tcHQgb2Yge3NlZW5bZGlnZXN0XX0iKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAg"
        "c2VlbltkaWdlc3RdID0gZXhhbXBsZS5pZA0KDQogICAgcmV0dXJuIGVycm9ycw0KDQoNCmRl"
        "ZiBfY2hlY2tfbmVhcl9kdXBsaWNhdGVzKGV4YW1wbGVzOiBsaXN0W0V4YW1wbGVdKSAtPiBs"
        "aXN0W3N0cl06DQogICAgIiIiRmxhZyBwcm9tcHRzIHRoYXQgYXJlIG5lYXJseSBpZGVudGlj"
        "YWwuDQoNCiAgICBSZXBvcnRlZCBhcyB3YXJuaW5ncywgbm90IGVycm9yczogdGVtcGxhdGVk"
        "IGdlbmVyYXRpb24gbGVnaXRpbWF0ZWx5DQogICAgcHJvZHVjZXMgc2ltaWxhciBwcm9tcHRz"
        "LCBhbmQgdGhlIHJpZ2h0IHJlc3BvbnNlIGlzIHVzdWFsbHkgdG8gd2lkZW4gdGhlDQogICAg"
        "Z2VuZXJhdG9yIHJhdGhlciB0aGFuIHRvIGRyb3AgZXhhbXBsZXMuDQogICAgIiIiDQogICAg"
        "d2FybmluZ3M6IGxpc3Rbc3RyXSA9IFtdDQogICAgdG9rZW5fc2V0cyA9IFsoZXhhbXBsZS5p"
        "ZCwgX3Rva2VucyhleGFtcGxlKSkgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXNdDQoNCiAgICBm"
        "b3IgaW5kZXgsIChsZWZ0X2lkLCBsZWZ0KSBpbiBlbnVtZXJhdGUodG9rZW5fc2V0cyk6DQog"
        "ICAgICAgIGZvciByaWdodF9pZCwgcmlnaHQgaW4gdG9rZW5fc2V0c1tpbmRleCArIDEgOiBp"
        "bmRleCArIDQwXToNCiAgICAgICAgICAgIGlmIG5vdCBsZWZ0IG9yIG5vdCByaWdodDoNCiAg"
        "ICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgc2ltaWxhcml0eSA9IGxlbihs"
        "ZWZ0ICYgcmlnaHQpIC8gbGVuKGxlZnQgfCByaWdodCkNCiAgICAgICAgICAgIGlmIHNpbWls"
        "YXJpdHkgPj0gTUFYX0pBQ0NBUkQ6DQogICAgICAgICAgICAgICAgd2FybmluZ3MuYXBwZW5k"
        "KA0KICAgICAgICAgICAgICAgICAgICBmIntsZWZ0X2lkfSBhbmQge3JpZ2h0X2lkfSBhcmUg"
        "e3NpbWlsYXJpdHk6LjAlfSBzaW1pbGFyIg0KICAgICAgICAgICAgICAgICkNCg0KICAgIHJl"
        "dHVybiB3YXJuaW5ncw0KDQoNCmRlZiBfY2hlY2tfY2hyb25vbG9naWNhbF9zcGxpdHMoZXhh"
        "bXBsZXM6IGxpc3RbRXhhbXBsZV0pIC0+IGxpc3Rbc3RyXToNCiAgICAiIiJIaXN0b3JpY2Fs"
        "IHNwbGl0cyBtdXN0IG5vdCBvdmVybGFwIGluIHRpbWUuDQoNCiAgICBJZiBhIHRlc3QgZXhh"
        "bXBsZSBwcmVkYXRlcyBhIHRyYWluaW5nIGV4YW1wbGUsIHRoZSBtb2RlbCBoYXMgYmVlbiB0"
        "cmFpbmVkDQogICAgb24gdGhlIGZ1dHVyZSBvZiBpdHMgb3duIHRlc3Qgc2V0IGFuZCBldmVy"
        "eSBtZXRyaWMgaXMgb3B0aW1pc3RpYy4NCiAgICAiIiINCiAgICBlcnJvcnM6IGxpc3Rbc3Ry"
        "XSA9IFtdDQogICAgYm91bmRzOiBkaWN0W3N0ciwgdHVwbGVbc3RyLCBzdHJdXSA9IHt9DQoN"
        "CiAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwgInRlc3QiKToNCiAg"
        "ICAgICAgc3RhbXBzID0gc29ydGVkKA0KICAgICAgICAgICAgZXhhbXBsZS5kZWNpc2lvbl90"
        "aW1lDQogICAgICAgICAgICBmb3IgZXhhbXBsZSBpbiBleGFtcGxlcw0KICAgICAgICAgICAg"
        "aWYgZXhhbXBsZS5zcGxpdCA9PSBzcGxpdCBhbmQgZXhhbXBsZS5wcm92ZW5hbmNlID09ICJo"
        "aXN0b3JpY2FsIg0KICAgICAgICApDQogICAgICAgIGlmIHN0YW1wczoNCiAgICAgICAgICAg"
        "IGJvdW5kc1tzcGxpdF0gPSAoc3RhbXBzWzBdLCBzdGFtcHNbLTFdKQ0KDQogICAgb3JkZXIg"
        "PSBbc3BsaXQgZm9yIHNwbGl0IGluICgidHJhaW4iLCAidmFsaWRhdGlvbiIsICJ0ZXN0Iikg"
        "aWYgc3BsaXQgaW4gYm91bmRzXQ0KICAgIGZvciBlYXJsaWVyLCBsYXRlciBpbiB6aXAob3Jk"
        "ZXIsIG9yZGVyWzE6XSk6DQogICAgICAgIGlmIHBhcnNlX3RpbWUoYm91bmRzW2xhdGVyXVsw"
        "XSkgPCBwYXJzZV90aW1lKGJvdW5kc1tlYXJsaWVyXVsxXSk6DQogICAgICAgICAgICBlcnJv"
        "cnMuYXBwZW5kKA0KICAgICAgICAgICAgICAgIGYiaGlzdG9yaWNhbCB7bGF0ZXJ9IHNwbGl0"
        "IHN0YXJ0cyBhdCB7Ym91bmRzW2xhdGVyXVswXX0sICINCiAgICAgICAgICAgICAgICBmImJl"
        "Zm9yZSB7ZWFybGllcn0gZW5kcyBhdCB7Ym91bmRzW2VhcmxpZXJdWzFdfSINCiAgICAgICAg"
        "ICAgICkNCg0KICAgIHJldHVybiBlcnJvcnMNCg0KDQpkZWYgX3N0YXRpc3RpY3MoZXhhbXBs"
        "ZXM6IGxpc3RbRXhhbXBsZV0pIC0+IGRpY3Rbc3RyLCBvYmplY3RdOg0KICAgIGRvbWFpbnMg"
        "PSBDb3VudGVyKGV4YW1wbGUuZG9tYWluLnZhbHVlIGZvciBleGFtcGxlIGluIGV4YW1wbGVz"
        "KQ0KICAgIHNwbGl0cyA9IENvdW50ZXIoZXhhbXBsZS5zcGxpdCBmb3IgZXhhbXBsZSBpbiBl"
        "eGFtcGxlcykNCiAgICBjaGFpbnMgPSBDb3VudGVyKGV4YW1wbGUuY2hhaW4gZm9yIGV4YW1w"
        "bGUgaW4gZXhhbXBsZXMgaWYgZXhhbXBsZS5jaGFpbikNCg0KICAgIHRyYWRpbmcgPSBbZSBm"
        "b3IgZSBpbiBleGFtcGxlcyBpZiBlLmRvbWFpbiBpcyBEb21haW4uVFJBRElOR19ERUNJU0lP"
        "Tl0NCiAgICBub19hY3Rpb24gPSBbDQogICAgICAgIGUgZm9yIGUgaW4gdHJhZGluZyBpZiBl"
        "LmV4cGVjdGVkX291dHB1dC5nZXQoImFjdGlvbiIpID09IFRyYWRlQWN0aW9uLk5PX0FDVElP"
        "Ti52YWx1ZQ0KICAgIF0NCg0KICAgIHN0YXRzOiBkaWN0W3N0ciwgb2JqZWN0XSA9IHsNCiAg"
        "ICAgICAgInRyYWRpbmdfZXhhbXBsZXMiOiBsZW4odHJhZGluZyksDQogICAgICAgICJub19h"
        "Y3Rpb25fZXhhbXBsZXMiOiBsZW4obm9fYWN0aW9uKSwNCiAgICAgICAgIm5vX2FjdGlvbl9y"
        "YXRpbyI6IChsZW4obm9fYWN0aW9uKSAvIGxlbih0cmFkaW5nKSkgaWYgdHJhZGluZyBlbHNl"
        "IDAuMCwNCiAgICB9DQoNCiAgICBmb3IgZG9tYWluLCBjb3VudCBpbiBkb21haW5zLml0ZW1z"
        "KCk6DQogICAgICAgIHN0YXRzW2YiZG9tYWluX3tkb21haW59Il0gPSBjb3VudA0KICAgIGZv"
        "ciBzcGxpdCwgY291bnQgaW4gc3BsaXRzLml0ZW1zKCk6DQogICAgICAgIHN0YXRzW2Yic3Bs"
        "aXRfe3NwbGl0fSJdID0gY291bnQNCiAgICBmb3IgY2hhaW4gaW4gQ0hBSU5TOg0KICAgICAg"
        "ICBzdGF0c1tmImNoYWluX3tjaGFpbn0iXSA9IGNoYWlucy5nZXQoY2hhaW4sIDApDQoNCiAg"
        "ICByZXR1cm4gc3RhdHMNCg0KDQpkZWYgX3Byb21wdF9oYXNoKGV4YW1wbGU6IEV4YW1wbGUp"
        "IC0+IHN0cjoNCiAgICBwcm9tcHQgPSAiXG4iLmpvaW4oDQogICAgICAgIGYie21lc3NhZ2Uu"
        "cm9sZX06e21lc3NhZ2UuY29udGVudH0iDQogICAgICAgIGZvciBtZXNzYWdlIGluIGV4YW1w"
        "bGUubWVzc2FnZXMNCiAgICAgICAgaWYgbWVzc2FnZS5yb2xlICE9ICJhc3Npc3RhbnQiDQog"
        "ICAgKQ0KICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihwcm9tcHQuZW5jb2RlKCJ1dGYtOCIp"
        "KS5oZXhkaWdlc3QoKQ0KDQoNCmRlZiBfdG9rZW5zKGV4YW1wbGU6IEV4YW1wbGUpIC0+IHNl"
        "dFtzdHJdOg0KICAgIHRleHQgPSAiICIuam9pbihtLmNvbnRlbnQgZm9yIG0gaW4gZXhhbXBs"
        "ZS5tZXNzYWdlcyBpZiBtLnJvbGUgPT0gInVzZXIiKQ0KICAgIHJldHVybiBzZXQodGV4dC5s"
        "b3dlcigpLnNwbGl0KCkpDQoNCg0KZGVmIGRhdGFzZXRfaGFzaChleGFtcGxlczogbGlzdFtF"
        "eGFtcGxlXSkgLT4gc3RyOg0KICAgICIiIkEgc3RhYmxlIGhhc2ggb2YgdGhlIHdob2xlIGRh"
        "dGFzZXQsIHJlY29yZGVkIGluIGV2ZXJ5IHRyYWluaW5nIHJ1bi4iIiINCiAgICBkaWdlc3Qg"
        "PSBoYXNobGliLnNoYTI1NigpDQogICAgZm9yIGV4YW1wbGUgaW4gc29ydGVkKGV4YW1wbGVz"
        "LCBrZXk9bGFtYmRhIGl0ZW06IGl0ZW0uaWQpOg0KICAgICAgICBkaWdlc3QudXBkYXRlKGV4"
        "YW1wbGUudG9fanNvbigpLmVuY29kZSgidXRmLTgiKSkNCiAgICByZXR1cm4gZGlnZXN0Lmhl"
        "eGRpZ2VzdCgpDQoNCg0KZGVmIG1haW4oKSAtPiBpbnQ6DQogICAgcGFyc2VyID0gYXJncGFy"
        "c2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlZhbGlkYXRlIGFuIEFUUkEtNEIgZGF0"
        "YXNldCIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiZGlyZWN0b3J5IiwgdHlwZT1QYXRo"
        "LCBoZWxwPSJkaXJlY3RvcnkgY29udGFpbmluZyAqLmpzb25sIikNCiAgICBhcmdzID0gcGFy"
        "c2VyLnBhcnNlX2FyZ3MoKQ0KDQogICAgZmlsZXMgPSBzb3J0ZWQoYXJncy5kaXJlY3Rvcnku"
        "Z2xvYigiKi5qc29ubCIpKQ0KICAgIGlmIG5vdCBmaWxlczoNCiAgICAgICAgcHJpbnQoZiJu"
        "byAuanNvbmwgZmlsZXMgZm91bmQgaW4ge2FyZ3MuZGlyZWN0b3J5fSIsIGZpbGU9c3lzLnN0"
        "ZGVycikNCiAgICAgICAgcmV0dXJuIDINCg0KICAgIGV4YW1wbGVzOiBsaXN0W0V4YW1wbGVd"
        "ID0gW10NCiAgICBmb3IgcGF0aCBpbiBmaWxlczoNCiAgICAgICAgZXhhbXBsZXMuZXh0ZW5k"
        "KGxvYWRfanNvbmwocGF0aCkpDQoNCiAgICByZXBvcnQgPSBjaGVjayhleGFtcGxlcykNCiAg"
        "ICBwcmludChyZXBvcnQucmVuZGVyKCkpDQogICAgcHJpbnQoZiJkYXRhc2V0IGhhc2g6IHtk"
        "YXRhc2V0X2hhc2goZXhhbXBsZXMpfSIpDQoNCiAgICByZXR1cm4gMCBpZiByZXBvcnQub2sg"
        "ZWxzZSAxDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICByYWlzZSBTeXN0"
        "ZW1FeGl0KG1haW4oKSkNCg=="
    ),
}

root = pathlib.Path('/kaggle/working/atra')
if root.exists():
    shutil.rmtree(root)

for name, blob in FILES.items():
    raw = base64.b64decode(blob)
    target = root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)
    print(f"{hashlib.sha256(raw).hexdigest()}  {name}  ({len(raw)} bytes)")

print()
print(sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()))

## 3. The trained adapter

`export.py` refuses an adapter whose training manifest is not `completed`,
so the manifest travels with it.

In [ ]:
import json, pathlib, shutil

# A dataset's layout inside /kaggle/input is not the folder that was uploaded,
# so the adapter is found by its config rather than by an assumed path.
inputs = pathlib.Path('/kaggle/input')
print('inputs:', [str(p) for p in inputs.glob('*')])
candidates = [p.parent for p in inputs.rglob('adapter_config.json')]
assert candidates, f'no adapter_config.json under {inputs}'
src = candidates[0]
print('adapter found at:', src)

dst = pathlib.Path('/kaggle/working/runs/atra-4b')
if dst.exists():
    shutil.rmtree(dst)
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst)
# Intermediate checkpoints are not needed for the merge.
for ckpt in dst.glob('checkpoint-*'):
    shutil.rmtree(ckpt)

print(sorted(p.name for p in dst.iterdir()))
manifest = json.loads((dst / 'manifest.json').read_text())
print('status      :', manifest['status'])
print('steps       :', manifest['steps'])
print('final loss  :', round(manifest['final_loss'], 6))
print('dataset hash:', manifest['dataset_hash'])
print('counts      :', manifest['dataset_counts'])
assert manifest['status'] == 'completed'

### 3.1 The chat template the GGUF does not carry

Attempt 2 of this kernel stopped at the probe: the prompt was 91 tokens with
or without `tools` in the request body. `evaluate.py` was sending the schemas
correctly — the server was discarding them, because the GGUF has no chat
template and llama-server fell back to a default that has no `tools` block.

The template is missing because transformers 4.57 writes it to a separate
`chat_template.jinja`, which is not among the adapter files listed above, so
`export.py`'s tokenizer had none to pass on to the converter. It is recovered
here from the base model and revision the training manifest names — the same
tokenizer `train.py` rendered all 800 training prompts with — and handed to
the server as a flag. The GGUF is not rebuilt: it stays byte-identical to the
exported artifact, sha256 `0c4ed0bb…d3348`.

In [ ]:
import hashlib, json, os, pathlib

# Why this cell exists.
#
# The second attempt at this evaluation stopped at the probe: the same prompt
# came back 91 tokens whether or not `tools` was in the request body. The fixed
# `evaluate.py` was sending the tool schemas correctly; the server was throwing
# them away.
#
# The chain is a lossy hand-off, one step at a time. `train.py` saves the
# tokenizer with `save_pretrained`, and transformers 4.57 writes the chat
# template to a separate `chat_template.jinja` file instead of into
# `tokenizer_config.json`. That file is not among the adapter's uploaded files
# (printed above: no `chat_template.jinja`). `export.py` rebuilds the tokenizer
# with `AutoTokenizer.from_pretrained(adapter)`, so it gets one with no
# template; `convert_hf_to_gguf.py` finds none to embed; and `llama-server`
# falls back to its built-in default, which has no `tools` block at all. The
# schemas were dropped in silence at the last step.
#
# So the first evaluation had two skews, not one: no tools, and not the
# template training used. The template is recovered here from the base model
# the training manifest names, at the revision it names — the same tokenizer
# `train.py:render_example` rendered all 800 training prompts with.
#
# The GGUF is deliberately left alone. Repairing the merge inputs would change
# the weights file and cost the published sha256 `0c4ed0bb…`; serving the
# recovered template is a flag on llama-server, and the artifact under
# evaluation stays byte-identical to the one that was exported.
os.environ.setdefault('HF_HOME', '/kaggle/temp/hf')
pathlib.Path('/kaggle/temp/hf').mkdir(parents=True, exist_ok=True)

from transformers import AutoTokenizer

BASE = manifest['base_model']
REVISION = manifest.get('base_revision')
print('base model   :', BASE)
print('base revision:', REVISION)

base_tokenizer = AutoTokenizer.from_pretrained(BASE, revision=REVISION)
template = base_tokenizer.chat_template
assert template, f'{BASE} carries no chat template either; nothing to serve with'

TEMPLATE_PATH = pathlib.Path('/kaggle/working/chat-template.jinja')
TEMPLATE_PATH.write_text(template, encoding='utf-8')
print('template     :', len(template), 'chars')
print('template sha :', hashlib.sha256(template.encode('utf-8')).hexdigest())

# Does this template honour `tools`? Rendered both ways, in the same shape
# train.py:render_example builds, so the answer is arithmetic rather than hope.
probe_messages = [
    {'role': 'system', 'content': 'system prompt'},
    {'role': 'user', 'content': 'user turn'},
]
probe_tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_market_snapshot',
            'description': 'Current price, liquidity and volume for a pool.',
            'parameters': {'type': 'object', 'required': ['chain'], 'properties': {'chain': {'type': 'string'}}},
        },
    }
]
plain = base_tokenizer.apply_chat_template(probe_messages, tokenize=False, add_generation_prompt=True)
with_tools = base_tokenizer.apply_chat_template(
    probe_messages, tools=probe_tools, tokenize=False, add_generation_prompt=True
)
print()
print('rendered without tools:', len(plain), 'chars')
print('rendered with tools   :', len(with_tools), 'chars')
print('tool name in rendering:', 'get_market_snapshot' in with_tools)
assert 'get_market_snapshot' not in plain
assert 'get_market_snapshot' in with_tools, (
    'the recovered template ignores `tools`; serving it would not fix anything'
)
print()
print(with_tools[: len(with_tools) - len(plain) + 200])

## 4. llama.cpp: converter, quantizer and server

`llama-quantize` makes the q4_k_m file and `llama-server` serves it. The
build prefers CUDA — the evaluation is ~95 generations of up to 512 tokens,
which is minutes on a T4 and hours on this machine's CPU — and falls back to
a CPU build rather than losing the run.

In [ ]:
%%bash
set -e
set -o pipefail
# /kaggle/temp does not exist unless a notebook creates it.
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf llama.cpp
git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
cd llama.cpp
echo "llama.cpp commit: $(git rev-parse HEAD)"
# Only the converter's own package. llama.cpp's requirements file pins a CPU
# build of torch, which would replace Kaggle's CUDA torch.
pip install -q --no-deps gguf

BUILD=/kaggle/temp/llama.cpp/build
set +e
ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d '. ')
set -e
if [ -z "${ARCH}" ]; then
  # A T4 is sm_75; the kernel is pinned to that machine shape.
  ARCH=75
fi
echo "cuda architecture: ${ARCH}"

# The server does the generating for ~95 evaluation prompts, so it is worth
# paying for a CUDA build; a CPU-only server would turn minutes into hours.
set +e
cmake -S /kaggle/temp/llama.cpp -B "$BUILD" -DCMAKE_BUILD_TYPE=Release \
  -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="${ARCH}" -DLLAMA_CURL=OFF \
  > /kaggle/temp/cmake-configure.log 2>&1
CFG=$?
BLD=1
if [ $CFG -eq 0 ]; then
  cmake --build "$BUILD" --target llama-quantize llama-server -j"$(nproc)" \
    > /kaggle/temp/cmake-build.log 2>&1
  BLD=$?
fi
set -e

if [ $CFG -ne 0 ] || [ $BLD -ne 0 ]; then
  echo "=== CUDA build failed (configure=$CFG build=$BLD); falling back to CPU ==="
  tail -40 /kaggle/temp/cmake-configure.log || true
  tail -40 /kaggle/temp/cmake-build.log 2>/dev/null || true
  rm -rf "$BUILD"
  cmake -S /kaggle/temp/llama.cpp -B "$BUILD" -DCMAKE_BUILD_TYPE=Release \
    -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /kaggle/temp/cmake-configure-cpu.log 2>&1
  cmake --build "$BUILD" --target llama-quantize llama-server -j"$(nproc)" \
    > /kaggle/temp/cmake-build-cpu.log 2>&1
  echo CPU > /kaggle/temp/build-kind.txt
else
  echo CUDA > /kaggle/temp/build-kind.txt
fi

echo "build kind: $(cat /kaggle/temp/build-kind.txt)"
ls -la "$BUILD/bin" | grep -E "llama-quantize|llama-server"

# The GGUF carries no chat template (see section 3.1), so the server has to be
# given one. Confirm this build takes the flag before the run depends on it.
echo
echo "--- llama-server chat template flags ---"
"$BUILD/bin/llama-server" --help 2>&1 | grep -E -- "--jinja|--chat-template-file" \
  || echo "WARNING: this build has neither --jinja nor --chat-template-file"

## 5. Merge and convert

In [ ]:
%%bash
set -e
set -o pipefail
# Keep the 8 GiB base download, the merged fp16 weights and the f16 GGUF
# intermediate off the 20 GiB /kaggle/working quota; only the q4_k_m file and
# its manifest are worth saving as output.
export HF_HOME=/kaggle/temp/hf
mkdir -p /kaggle/temp/hf
cd /kaggle/working/atra
python export.py --adapter /kaggle/working/runs/atra-4b --out /kaggle/temp/atra-export \
  --llama-cpp /kaggle/temp/llama.cpp --gguf q4_k_m 2>&1 | tail -40

ls -la /kaggle/temp/atra-export
cp /kaggle/temp/atra-export/atra-4b-q4_k_m.gguf /kaggle/working/
cp /kaggle/temp/atra-export/Modelfile /kaggle/working/
cp /kaggle/temp/atra-export/export-manifest.json /kaggle/working/
# The merged fp16 copy has done its job once the GGUF exists.
rm -rf /kaggle/temp/atra-export/merged
cat /kaggle/working/export-manifest.json
df -h /kaggle/working

## 6. The test set

Rebuilt deterministically, with the seed and per-domain count the thresholds
were written for, rather than carried: the build is the specification.

In [ ]:
%%bash
set -e
set -o pipefail
cd /kaggle/working/atra
# The same command, seed and per-domain count the thresholds were written for.
python -m data.build --seed 42 --per-domain 200 --out data/out
python -m data.checks data/out > /kaggle/working/data-checks.txt 2>&1 || true
tail -32 /kaggle/working/data-checks.txt
echo
wc -l data/out/*.jsonl
sha256sum data/out/test.jsonl

## 7. Serve the GGUF

In [ ]:
import json, os, pathlib, subprocess, time, urllib.request

BIN = '/kaggle/temp/llama.cpp/build/bin/llama-server'
GGUF = '/kaggle/working/atra-4b-q4_k_m.gguf'
PORT = 8080
ENDPOINT = f'http://127.0.0.1:{PORT}'
LOG = pathlib.Path('/kaggle/working/llama-server.log')
BUILD_KIND = pathlib.Path('/kaggle/temp/build-kind.txt').read_text().strip()
print('build kind:', BUILD_KIND)

server = None


def start(extra):
    global server
    command = [
        BIN, '-m', GGUF,
        '--host', '127.0.0.1', '--port', str(PORT),
        '-c', '8192', '-ngl', '99', '--parallel', '1',
        '-t', str(os.cpu_count() or 4),
    ] + extra
    print('$', ' '.join(command))
    handle = LOG.open('ab')
    server = subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT)
    return server


def wait_ready(timeout=900):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if server.poll() is not None:
            print('server exited with', server.returncode)
            return False
        try:
            with urllib.request.urlopen(ENDPOINT + '/health', timeout=5) as response:
                if response.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(3)
    return False


ready = False
# --jinja makes the server render a jinja chat template rather than its own
# hardcoded formats, and --chat-template-file supplies the one the converter
# could not embed: the training tokenizer's, recovered in section 3.1. Without
# it the server falls back to a default template that has no `tools` block,
# which is exactly how the tool schemas went missing.
#
# The fallbacks are there so a flag this build does not have loses the flag
# rather than the run; the probe in section 8 then fails loudly, because a run
# served without the tool block is the bug this kernel exists to fix.
for extra in (
    ['--jinja', '--chat-template-file', str(TEMPLATE_PATH)],
    ['--jinja'],
    [],
):
    start(extra)
    ready = wait_ready()
    if ready:
        print('server ready; extra args:', extra)
        SERVER_ARGS = extra
        break
    print('server not ready with', extra)
    try:
        server.kill()
        server.wait(timeout=30)
    except Exception:
        pass
    print(LOG.read_text(errors='replace')[-4000:])

assert ready, 'llama-server never answered /health'
print(LOG.read_text(errors='replace')[-1500:])

## 8. What the served model actually says

First the prompt-shape check: the same tool_use prompt sent with and without
the tool block, so the token counts show what the fix changed. The run stops
here if the tools do not lengthen the prompt, or if llama-server swallows the
reply into `tool_calls` and returns empty content — either would make every
metric below a statement about the harness rather than the model.

Then one prompt per domain, printed in full next to the expected answer. If a
metric fails later, this is the evidence for why.

In [ ]:
import json, pathlib, time, urllib.request

TEST = pathlib.Path('/kaggle/working/atra/data/out/test.jsonl')
examples = [json.loads(line) for line in TEST.read_text(encoding='utf-8').splitlines() if line.strip()]
print('test examples:', len(examples))


def tools_for(example):
    """The tool block, in the exact shape EndpointModel.generate builds.

    The first evaluation posted only `messages`. Every training prompt was
    rendered with `apply_chat_template(messages, tools=...)`, so the model was
    judged on a prompt shape it had never seen and tool_selection_accuracy came
    back a clean 0/8. The probe has to send what the harness sends, otherwise
    it is evidence for a different run than the one being scored.
    """
    return [
        {
            'type': 'function',
            'function': {
                'name': tool['name'],
                'description': tool['description'],
                'parameters': tool['parameters'],
            },
        }
        for tool in example.get('tools', [])
    ]


def ask(messages, tools=None, timeout=300):
    body = {
        'model': 'atra-4b-v0',
        'messages': messages,
        'temperature': 0.0,
        'max_tokens': 512,
        'stream': False,
    }
    if tools:
        body['tools'] = tools
    payload = json.dumps(body).encode('utf-8')
    request = urllib.request.Request(
        ENDPOINT + '/v1/chat/completions',
        data=payload,
        headers={'content-type': 'application/json'},
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read())


def prompt_of(example):
    return [
        {'role': message['role'], 'content': message['content']}
        for message in example['messages'] if message['role'] != 'assistant'
    ]


# ---------------------------------------------------------------------------
# Fail fast on the one thing that would make the whole run meaningless.
#
# Two ways the number could be a lie. Either the tools never reach the model —
# the bug this re-run exists to fix, visible as a prompt of ~100 tokens instead
# of several hundred — or llama-server parses the reply as a native tool call
# and hands back `content: null`, which evaluate.py reads as the empty string
# and scores as unparseable. Both are checked here, before ~95 generations.
# ---------------------------------------------------------------------------
tool_example = next(e for e in examples if e['domain'] == 'tool_use')
with_tools = ask(prompt_of(tool_example), tools_for(tool_example))
without_tools = ask(prompt_of(tool_example), None)

n_with = with_tools.get('usage', {}).get('prompt_tokens')
n_without = without_tools.get('usage', {}).get('prompt_tokens')
message = with_tools['choices'][0]['message']

print('=' * 78)
print('PROMPT SHAPE CHECK  (example:', tool_example['id'], ')')
print('  prompt_tokens WITHOUT tools :', n_without, '  <- what the first, broken run sent')
print('  prompt_tokens WITH tools    :', n_with, '  <- what the fixed harness sends')
print('  difference                  :', (n_with - n_without) if (n_with and n_without) else '?')
print('  tools in the example        :', [t['name'] for t in tool_example.get('tools', [])])
print('  reply message keys          :', sorted(message.keys()))
print('  content is None             :', message.get('content') is None)
print('  tool_calls present          :', bool(message.get('tool_calls')))
print('  raw message                 :', json.dumps(message, sort_keys=True)[:1200])
print('=' * 78)

assert n_with and n_without and n_with > n_without, (
    f'the tool block did not change the prompt length ({n_without} -> {n_with}); '
    'the server is not rendering tools and the run would repeat the original bug'
)
# A reply swallowed into `tool_calls` leaves evaluate.py with "" and scores as
# unparseable output. That is a serving artefact, not a model result, and it
# must not be reported as one.
assert not (message.get('content') in (None, '') and message.get('tool_calls')), (
    'llama-server parsed the reply into tool_calls and returned empty content; '
    'evaluate.py reads message.content, so every metric would be a floor value'
)

# One example per domain, printed in full next to the expected answer. If a
# metric fails later, this is the evidence for why.
seen = set()
probes = []
for example in examples:
    if example['domain'] not in seen:
        seen.add(example['domain'])
        probes.append(example)

for example in probes:
    messages = prompt_of(example)
    started = time.time()
    body = ask(messages, tools_for(example))
    elapsed = time.time() - started
    usage = body.get('usage', {})
    print('=' * 78)
    print('id       :', example['id'], '| domain:', example['domain'], '| tags:', example.get('tags'))
    print('user     :', messages[-1]['content'][:300])
    print('expected :', json.dumps(example['expected_output'], sort_keys=True)[:400])
    print('reply    :', (body['choices'][0]['message']['content'] or '')[:900])
    print(f'timing   : {elapsed:.1f}s  usage={usage}')

## 9. Evaluate

In [ ]:
%%bash
# Deliberately no `set -e`: evaluate.py exits non-zero when a threshold is
# missed, and a missed threshold is a result to report, not a kernel failure.
set -o pipefail
cd /kaggle/working/atra
python evaluate.py \
  --data data/out/test.jsonl \
  --model http://127.0.0.1:8080 \
  --model-name atra-4b-v0 \
  --out /kaggle/working/eval.json \
  --config config/default.yaml 2>&1 | tee /kaggle/working/evaluate-stdout.txt
CODE=${PIPESTATUS[0]}
echo
echo "evaluate.py exit code: ${CODE}"
echo "${CODE}" > /kaggle/working/evaluate-exit-code.txt

## 10. Did the tools reach the model?

The probe proves it for six requests. This proves it for all of them, from
llama-server's own record of how long each prompt was.

In [ ]:
import pathlib, re, statistics

# Direct evidence that the tool block reached the model on every scored
# request, not just on the probes. llama-server records the full prompt length
# per request — cached prefix included — so this counts what the model was
# actually shown across the whole evaluation.
log = pathlib.Path('/kaggle/working/llama-server.log').read_text(errors='replace')

counts = [int(n) for n in re.findall(r'n_prompt_tokens\s*=\s*(\d+)', log)]
source = 'n_prompt_tokens'
if not counts:
    # Older server builds only print the timing line, which counts the tokens
    # actually evaluated rather than the whole prompt; under prompt caching
    # that under-reports, so it is a fallback and labelled as one.
    counts = [int(n) for n in re.findall(r'prompt eval time.*?/\s*(\d+)\s*tokens', log)]
    source = 'prompt eval time (uncached tokens only)'

print('PROMPT LENGTH ACROSS EVERY REQUEST  (source:', source + ')')
print('requests logged :', len(counts))
if counts:
    print('min             :', min(counts))
    print('median          :', int(statistics.median(counts)))
    print('max             :', max(counts))
    print('mean            :', round(statistics.fmean(counts), 1))
    print()
    print('first 12        :', counts[:12])
    print('last 12         :', counts[-12:])
    print()
    print('below 200 tokens:', sum(1 for n in counts if n < 200),
          '  <- the first, broken run sat at 91-119 on every request')
else:
    print('no prompt-length lines in the server log; rely on the probe usage figures above')

print()
for line in log.splitlines():
    if 'n_prompt_tokens' in line:
        print('sample:', line.strip()[:200])
        break

## 11. Result

In [ ]:
import json, pathlib

report = json.loads(pathlib.Path('/kaggle/working/eval.json').read_text(encoding='utf-8'))
code = pathlib.Path('/kaggle/working/evaluate-exit-code.txt').read_text().strip()

print('=' * 100)
print('ATRA-4B EVALUATION RESULT')
print('=' * 100)
print('model        :', report['model'])
print('dataset hash :', report['dataset_hash'])
print('examples     :', report['examples'])
print('ran at       :', report['ran_at'], f"({report['duration_sec']}s)")
print('gguf         : /kaggle/working/atra-4b-q4_k_m.gguf')
print('json         : /kaggle/working/eval.json')
print()

header = f"{'metric':<30}{'score':>9}{'threshold':>12}{'direction':>16}{'passed':>12}{'verdict':>10}{'margin':>12}"
print(header)
print('-' * len(header))

missed = []
for metric in report['metrics']:
    threshold = metric['threshold']
    lower = metric['lower_is_better']
    direction = 'lower is better' if lower else 'higher is better'
    fraction = f"{metric['passed']}/{metric['total']}"
    if threshold is None:
        verdict, margin = 'n/a', ''
    else:
        ok = metric['meets_threshold']
        verdict = 'PASS' if ok else 'FAIL'
        delta = (threshold - metric['score']) if lower else (metric['score'] - threshold)
        margin = f"{delta:+.4f}"
        if not ok:
            missed.append((metric['name'], metric['score'], threshold, delta, lower))
    shown = 'n/a' if threshold is None else f'{threshold:.4f}'
    print(f"{metric['name']:<30}{metric['score']:>9.4f}{shown:>12}{direction:>16}{fraction:>12}{verdict:>10}{margin:>12}")

print()
if missed:
    print(f'{len(missed)} metric(s) BELOW THRESHOLD:')
    for name, score, threshold, delta, lower in missed:
        word = 'above the maximum by' if lower else 'short of the minimum by'
        print(f'  - {name}: {score:.4f} vs {threshold:.4f} ({word} {abs(delta):.4f})')
else:
    print('every metric meets its threshold.')

print()
print('evaluate.py exit code:', code)
print('OVERALL:', 'PASS' if code == '0' else 'FAIL')
print()
print('example ids that failed each metric (evaluate.py records the first ten):')
for metric in report['metrics']:
    if metric['failures']:
        print(f"  {metric['name']}: {metric['failures']}")

print()
print('=' * 100)
print('raw eval.json')
print('=' * 100)
print(json.dumps(report, indent=2, sort_keys=True))

## 12. What is kept

- `eval.json` — the evaluation result, thresholds included
- `atra-4b-q4_k_m.gguf` + `Modelfile` — the exact artifact that was evaluated
- `export-manifest.json` — adapter, base revision and training manifest
- `evaluate-stdout.txt`, `evaluate-exit-code.txt`, `data-checks.txt`

Whether the repository's UNTRAINED label changes is decided by the numbers
above, and not in this notebook.

In [ ]:
%%bash
set +e
pkill -f llama-server
sleep 3
# The server log is useful but not an artifact; keep only its tail.
tail -c 200000 /kaggle/working/llama-server.log > /kaggle/working/llama-server.tail.log 2>/dev/null
rm -f /kaggle/working/llama-server.log
rm -rf /kaggle/working/runs /kaggle/working/atra/data/out/train.jsonl
ls -la /kaggle/working
du -sh /kaggle/working
echo "done"